# **Thông tin nhóm**
- Lớp: ML Labs 23KHDL1
- Nhóm: 4
- Sinh viên: 
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Thu thập dữ liệu**

# **1. Tổng quan**
Notebook này trình bày toàn bộ quá trình thu thập dữ liệu phục vụ cho bài toán phân tích và dự đoán trong lĩnh vực du lịch - dịch vụ tại **Thành phố Hồ Chí Minh**. Dữ liệu được thu thập từ **3 nguồn** khác nhau, bao phủ **3 nhóm địa điểm** chính:
- **Google Maps**: các địa điểm tham quan (di tích, chùa, công viên, khu vui chơi, ...)
- **Booking**: các địa điểm lưu trú (khách sạn, resort, nhà nghỉ, homestay, ...)
- **Foody**: các địa điểm ăn uống (nhà hàng, quán ăn, cafe, ...)

## 1.2 Quyền riêng tư và bản quyền dữ liệu

In [ ]:
import urllib.robotparser

rp = urllib.robotparser.RobotFileParser()
rp.set_url('https://www.google.com/maps/robots.txt')
rp.read()
print("Google Maps: ", rp.can_fetch('*', 'https://www.google.com/maps'))

rp.set_url('https://booking.com/robots.txt')
rp.read()
print("Booking.com: ", rp.can_fetch('*', 'https://booking.com'))

rp.set_url('https://www.foody.vn/robots.txt')
rp.read()
print("Foody.vn: ", rp.can_fetch('*', 'https://www.foody.vn'))

Google Maps:  True
Booking.com:  True
Foody.vn:  True


# **2. Google Maps**


## 2.1 Giới thiệu

**Mục tiêu**: Thu thập dữ liệu ~2000 địa điểm tham quan tại Thành phố Hồ Chí Minh

**Phân loại**: có nhãn category là `attraction`, bao gồm: di tích, bảo tàng, công viên, chùa, nhà thờ, khu vui chơi, TTTM, chợ, spa, gym, sở thú, thư viện, rạp phim, karaoke,...

**Phương pháp tiếp cận**: Nhóm sử dụng phương pháp **Web Scraping** thông qua giả lập trình duyệt với thư viện `Selenium` để thu thập dữ liệu từ Google Maps.

**Kết quả** (`ggmaps.csv` / `ggmaps.json`):
| Cột | Kiểu dữ liệu | Mô tả |
|-----|------|-------|
| name | str | Tên địa điểm |
| address | str | Địa chỉ |
| star | float | Đánh giá sao (0-5) |
| nums_comments | int | Tổng số lượt đánh giá |
| price | int | Giá vé |
| category | str | Phân loại nhóm |
| hours | str | Giờ hoạt động |
| comments | list str | Danh sách đánh giá |
| comment_scores | list float | Danh sách điểm đánh giá |
| url | str | Đường dẫn trên GoogleMap |

## 2.2 Thư viện

In [16]:
%pip install selenium webdriver-manager pandas tqdm ipywidgets -q

import os
import re
import json
import time
import random
import pandas as pd

from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    ElementClickInterceptedException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager

CONFIG = {
    'headless': False,              # True = chạy ẩn, False = thấy trình duyệt
    # 'max_places_per_query': 200,    # Số địa điểm tối đa mỗi query
    'max_places_per_query': 200,    
    'max_reviews_per_place': 50,    # Số reviews tối đa mỗi địa điểm
    'years_back': 5,                # Lấy reviews trong N năm trở lại
    'sleep_min': 1.5,               # Delay tối thiểu (giây)
    'sleep_max': 3.5,               # Delay tối đa (giây)
    'sleep_between_places': 1,      # Delay giữa các địa điểm
    'sleep_between_queries': 2,     # Delay giữa các query
    'output_dir': 'data',           # Thư mục lưu dữ liệu
    'checkpoint_interval': 10,      # Lưu checkpoint mỗi N địa điểm
    'disable_images': False,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2.3 Danh sách tìm kiếm

Để thu thập dữ liệu đa dạng và bao phủ rộng các loại địa điểm tham quan tại TP. Hồ Chí Minh, nhóm xây dựng một danh sách các câu truy vấn (search queries) được sử dụng để tìm kiếm trên Google Maps (có thể mở rộng).

Cách thiết kế danh sách `SEARCH_QUERIES`:
- Tìm kiếm theo các địa điểm
- Tìm kiếm theo khu vực địa lý
- Sử dụng cả tiếng Anh - Việt

In [17]:
SEARCH_QUERIES = {
    'attraction': [
        'historical sites in Ho Chi Minh City',
        'museums in Ho Chi Minh City',
        'bảo tàng gần đây Sài Gòn',
        'di tích lịch sử gần đây Sài Gòn',

        'temples in Ho Chi Minh City',
        'pagoda in Ho Chi Minh City',
        'churches in Ho Chi Minh City',
        'chùa gần đây Sài Gòn',
        'nhà thờ gần đây Sài Gòn',

        'parks in Ho Chi Minh City',
        'gardens in Ho Chi Minh City',
        'công viên gần đây Sài Gòn',

        'amusement parks in Ho Chi Minh City',
        'water parks in Ho Chi Minh City',
        'entertainment centers in Ho Chi Minh City',
        'khu vui chơi gần đây Sài Gòn',
        'công viên nước gần đây Sài Gòn',

        'cinemas in Ho Chi Minh City',
        'theaters in Ho Chi Minh City',
        'rạp chiếu phim gần đây Sài Gòn',

        'shopping malls in Ho Chi Minh City',
        'markets in Ho Chi Minh City',
        'trung tâm thương mại gần đây Sài Gòn',
        'chợ gần đây Sài Gòn',

        'spa in Ho Chi Minh City',
        'massage in Ho Chi Minh City',
        'spa gần đây Sài Gòn',

        'gyms in Ho Chi Minh City',
        'sports centers in Ho Chi Minh City',
        'phòng gym gần đây Sài Gòn',
        'sân bóng gần đây Sài Gòn',

        'zoo in Ho Chi Minh City',
        'aquarium in Ho Chi Minh City',

        'libraries in Ho Chi Minh City',
        'thư viện gần đây Sài Gòn',

        'karaoke in Ho Chi Minh City',
        'karaoke gần đây Sài Gòn',

        'places to visit in District 1 Ho Chi Minh',
        'places to visit in District 2 Ho Chi Minh',
        'places to visit in District 3 Ho Chi Minh',
        'places to visit in District 4 Ho Chi Minh',
        'places to visit in District 5 Ho Chi Minh',
        'places to visit in District 6 Ho Chi Minh',
        'places to visit in District 7 Ho Chi Minh',
        'places to visit in District 8 Ho Chi Minh',
        'places to visit in District 9 Ho Chi Minh',
        'places to visit in District 10 Ho Chi Minh',
        'places to visit in District 11 Ho Chi Minh',
        'places to visit in District 12 Ho Chi Minh',
        'places to visit in Go Vap Ho Chi Minh',
        'places to visit in Binh Thanh Ho Chi Minh',
        'places to visit in Phu Nhuan Ho Chi Minh',
        'places to visit in Tan Phu Ho Chi Minh',
        'places to visit in Tan Binh Ho Chi Minh',
        'places to visit in Binh Tan Ho Chi Minh',
        'places to visit in Hoc Mon Ho Chi Minh',
        'places to visit in Cu Chi Ho Chi Minh',
        'places to visit in Binh Chanh Ho Chi Minh',
        'places to visit in Can Gio Ho Chi Minh',
        'places to visit in Nha Be Ho Chi Minh',
        'places to visit in Thu Duc Ho Chi Minh',

        'tourist attractions in Ho Chi Minh City',
        'things to do in Ho Chi Minh City',
        'best places to visit in Saigon',
        'top attractions Saigon',
        'điểm tham quan nổi tiếng Sài Gòn',
        'địa điểm du lịch gần đây Sài Gòn',
    ],
}

total_queries = len(SEARCH_QUERIES['attraction'])
print(f"Tổng số queries: {total_queries}")
print(f"Số lượng dữ liệu ước tính: {total_queries} queries × {CONFIG['max_places_per_query']} places = {total_queries * CONFIG['max_places_per_query']} (trước khi khử trùng lập)")

Tổng số queries: 67
Số lượng dữ liệu ước tính: 67 queries × 200 places = 13400 (trước khi khử trùng lập)


## 2.4 Thu thập dữ liệu
Nhóm khởi tạo một trình duyệt Chrome thực sự và điều khiển nó tự động thông qua việc mô phỏng hoàn toàn các thao tác của người dùng thật: nhập từ khóa tìm kiếm, cuộn danh sách kết quả, nhấn vào từng địa điểm, chuyển tab, và đọc nội dung hiển thị trên trang.

Để tránh bị Google phát hiện và chặn dưới dạng bot, nhóm áp dụng một số kỹ thuật chống phát hiện:
- Ẩn dấu hiệu tự động hóa bằng cách ghi đè thuộc tính `navigator.webdriver` về `undefined`, loại bỏ cờ `enable-automation` và tắt `useAutomationExtension`
- Thiết lập đồ trễ ngẫu nhiên giữa mỗi thao tác nhầm mô phỏng tốc độ tương tác tự nhiên của con người thay vì gửi request liên tục với tần suất cố định.
- Cài đặt cơ chế checkpoint để lưu dữ liệu định kỳ, đảm bảo không mất dữ liệu nếu quá trình thu thập bị gián đoạn do lỗi mạng, bị chặn, hoặc người dùng dừng thủ công.

## 2.4.1 Khai báo lớp

Lớp `GoogleMapsScraper` bao gồm các nhóm phương thức chính: **Khởi tạo & Tiện ích**, **Tìm kiếm & Cuộn**, **Trích xuất dữ liệu** và **Lưu trữ**

- Khởi tạo và tiện ích:
    - Nhận cầu hình từ `CONFIG`, khởi tạo danh sách để lưu trữ kể quả và chuẩn bị trình duyệt
    - Cấu hình và khởi tạo trình duyệt với các thiết lập: tự động hóa, tùy chọn hiệu năng, quản lý phiên bản, chấp nhận cookie

- Tìm kiếm và cuộn danh sách: 
    - Xây dựng URL tìm kiếm với các query đã mã hóa với tọa độ trong tâm Thành phố Hồ Chí Minh (mức zoom 12)
    - Cuộn danh sách kết quả để tải thêm địa điểm đến khi đạt đủ số lượng hoặc đến cuối danh sách
    - Tạo thành danh sách URL với mỗi đường dẫn tương ứng với một địa điểm trên Google Maps
- Trích xuất dữ liệu:
    - Mã định danh duy nhất dùng cho việc khử trùng lặp 
    - `name`: tên địa điểm từ thẻ tiêu đề
    - `star`: sao đánh giá 
    - `nums_comments`: tổng số lượt đánh giá
    - `price`: giá vé vào cửa (đã quy sang VND), được trích xuất với 3 cách tiếp cận khác nhau
    - `address`: địa chỉ, được trích xuất với 5 cách tiếp cận khác nhau  
    - `category`: nhãn phân loại (được gán nhãn `attraction`)
    - `hours`: giờ hoạt động, được trích xuất với 2 cách tiếp cận khác nhau
    - `comments`: danh sách đánh giá
    - `nums_comments`: danh sách điểm đán giá
    - `url`: đường dẫn Google Maps 
- Lưu trữ: 
    - Lưu dữ liệu tạm vào file checkpoint.csv sau `N` địa điểm
    - Lưu kết quả cuối cùng ở 2 định dạng: **.csv** (`ggmap.csv`) dùng cho phân tích dạng bảng và **.json** (`ggmap.json`) dùng cho trao đổi dữ liệu

In [ ]:
COLUMN_ORDER = ['name', 'address', 'star', 'nums_comments', 'price', 'category', 'hours', 
                'comments', 'comment_scores', 'url']

class GoogleMapsScraper:
    def __init__(self, config=None):
        self.config = config or CONFIG
        self.all_data = []
        self.seen_place_ids = set()
        self._setup_driver()

    # Khởi tạo và tiện ích
    def _setup_driver(self):
        options = Options()
        options.add_argument('--lang=vi')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--window-size=1920,1080')
        
        if self.config.get('disable_images', True):
            prefs = {"profile.managed_default_content_settings.images": 2}
            options.add_experimental_option("prefs", prefs)
        
        options.add_argument('--disable-gpu') 
        options.add_argument('--disable-extensions')
        options.add_argument('--dns-prefetch-disable')
        options.add_experimental_option('excludeSwitches', ['enable-automation'])
        options.add_experimental_option('useAutomationExtension', False)

        if self.config.get('headless', False):
            options.add_argument('--headless=new')

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        
        self.driver.execute_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
        
        self.wait = WebDriverWait(self.driver, 10)

    def _random_sleep(self, min_sec=None, max_sec=None):
        lo = min_sec or self.config['sleep_min']
        hi = max_sec or self.config['sleep_max']
        time.sleep(random.uniform(lo, hi))

    def _accept_cookies(self):
        selectors = [
            'button[aria-label*="Accept"]',
            'button[aria-label*="Chấp nhận"]',
            'button[aria-label*="Đồng ý"]',
            'form[action*="consent"] button',
            'button[jsname="b3VHJd"]',
        ]
        for sel in selectors:
            try:
                btn = WebDriverWait(self.driver, 3).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
                )
                btn.click()
                time.sleep(1)
                return
            except TimeoutException:
                continue

    # Tìm kiếm và cuộn
    def search_places(self, query):
        import urllib.parse

        encoded_query = urllib.parse.quote(query)
        search_url = (
            f"https://www.google.com/maps/search/{encoded_query}/"
            f"@10.8231,106.6297,12z"
        )
        
        self.driver.get(search_url)
        self._random_sleep(1, 2)
        self._accept_cookies()

        # Kiểm tra có phải là 1 địa điểm cụ thể hay không (không phải danh sách kết quả)
        current_url = self.driver.current_url
        if '/place/' in current_url and '/search/' not in current_url:
            print(f"    [INFO] Redirected to single place, trying search box...")
            try:
                search_box = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR,
                        'input#searchboxinput, input[name="q"], '
                        'input[aria-label*="Search"], input[aria-label*="Tìm kiếm"]'))
                )
                search_box.clear()
                time.sleep(0.5)
                search_box.send_keys(query)
                time.sleep(0.5)
                search_box.send_keys(Keys.ENTER)
                self._random_sleep(2, 3)
            except Exception as e:
                print(f"    [WARN] Search box fallback failed: {e}")

        # Hiển thị danh sách kết quả
        try:
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )
            return True
        except TimeoutException:
            print(f"Không thấy danh sách kết quả cho: {query}")
            try:
                WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
                )
                return True
            except TimeoutException:
                print(f"Bỏ qua query: {query}")
                return False

    def scroll_results(self, target_count=None):
        if target_count is None:
            target_count = self.config['max_places_per_query']

        try:
            # Xác định vùng cuộn 
            feed = self.wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )
        except TimeoutException:
            print("Không tìm thấy results feed")
            return 0

        last_count = 0
        no_change_count = 0

        # Thực hiện cuốn để đếm số lượng dữ liệu
        while no_change_count < 5:
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollHeight', feed
            )
            self._random_sleep(0.5, 1)

            results = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')
            current_count = len(results)

            if current_count >= target_count:
                break

            if current_count == last_count:
                no_change_count += 1
                try:
                    end_el = self.driver.find_element(By.XPATH,
                        '//span[contains(text(),"end of the list") or '
                        'contains(text(),"cuối danh sách") or '
                        'contains(text(),"xem hết")]')
                    if end_el.is_displayed():
                        break
                except NoSuchElementException:
                    pass
            else:
                no_change_count = 0
            last_count = current_count

        final = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')

        return len(final)

    def get_place_urls(self):
        urls = []

        try:
            feed = self.driver.find_element(By.CSS_SELECTOR, 'div[role="feed"]')
            links = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')

            for link in links:
                href = link.get_attribute('href')
                if href:
                    urls.append(href)
        except Exception as e:
            print(f"Lỗi lấy URLs: {e}")
            
        return urls

    # Trích xuất dữ liệu
    def _safe_text(self, by, selector, default=''):
        try:
            return self.driver.find_element(by, selector).text.strip()
        except (NoSuchElementException, StaleElementReferenceException):
            return default

    def _safe_attr(self, by, selector, attr, default=''):
        try:
            return self.driver.find_element(by, selector).get_attribute(attr)
        except (NoSuchElementException, StaleElementReferenceException):
            return default

    def _extract_place_id(self, url):
        # ID nội bộ của GoogleMap
        match = re.search(r'!1s(0x[a-f0-9]+:0x[a-f0-9]+)', url)
        if match:
            return match.group(1)
        
        # ID trên url
        match = re.search(r'/place/([^/]+)/', url)
        if match:
            return match.group(1)
        
        # Giữ nguyên
        return url

    def extract_place(self, url, pid):
        place_id = pid

        try:
            self.driver.get(url)
            self._random_sleep(1, 2)

            record = {}

            # Name (str)
            record['name'] = self._safe_text(By.CSS_SELECTOR, 'h1.DUwDvf')
            if not record['name']:
                record['name'] = self._safe_text(By.CSS_SELECTOR, 'h1')

            # Star (float)
            rating_text = self._safe_text(
                By.CSS_SELECTOR, 'div.F7nice span[aria-hidden="true"]'
            )
            try:
                record['star'] = float(rating_text.replace(',', '.'))
            except (ValueError, AttributeError):
                record['star'] = None

            # Nums_comments (int)
            record['nums_comments'] = self._extract_nums_comments()

            # Price (int)
            record['price'] = self._extract_price()

            # Address (str)
            record['address'] = self._extract_address()

            # Category (str)
            record['category'] = 'attraction'

            # Hours (str) 
            record['hours'] = self._extract_hours()

            # Comments (list of str) + Comment scores (list of float) 
            comments_data = self._get_comments(url, place_id)
            record['comments'] = comments_data['comments']
            record['comment_scores'] = comments_data['scores']

            # URL (str)
            record['url'] = url

            self.seen_place_ids.add(place_id)

            return record

        except Exception as e:
            print(f"Lỗi trong việc extract_place: {e}")
            return None

    def _extract_address(self):     
        '''
        Đảm bảo ở trong tab Tổng quan
        '''
        # Cách 1: Các selector phổ biến nhất
        address_selectors = [
            (By.CSS_SELECTOR, 'button[data-item-id="address"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id="address"] div.fontBodyMedium'),
            (By.CSS_SELECTOR, 'button[data-item-id="oloc"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id="oloc"] div.fontBodyMedium'),
            (By.CSS_SELECTOR, 'button[data-item-id*="laddress"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id^="address"] div.Io6YTe'),
        ]
        for by, sel in address_selectors:
            try:
                el = self.driver.find_element(by, sel)
                text = el.text.strip()
                if text:
                    return text
            except (NoSuchElementException, StaleElementReferenceException):
                continue

        # Cách 2: Tìm qua aria-label chứa "Địa chỉ"
        try:
            el = self.driver.find_element(By.XPATH,
                '//button[contains(@aria-label,"Địa chỉ:") or contains(@aria-label,"Address:")]')
            aria = el.get_attribute('aria-label')
            if aria:
                for prefix in ['Địa chỉ:', 'Address:']:
                    if prefix in aria:
                        addr = aria.split(prefix, 1)[1].strip()
                        if addr:
                            return addr
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 3: Tìm icon location pin -> lấy text
        try:
            addr_el = self.driver.find_element(By.XPATH,
                '//button[@data-tooltip and @data-tooltip="Sao chép địa chỉ"]//div[contains(@class,"Io6YTe")]'
                ' | //button[@data-tooltip and @data-tooltip="Copy address"]//div[contains(@class,"Io6YTe")]')
            text = addr_el.text.strip()
            if text:
                return text
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 4: Scroll xuống để tìm address (có thể bị ẩn ở dưới)
        try:
            scrollable = self.driver.find_element(By.CSS_SELECTOR, 'div.m6QErb.DxyBCb.kA9KIf.dS8AEf')
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollTop + 300', scrollable
            )
            self._random_sleep(0, 0.5)

            for by, sel in address_selectors[:2]:
                try:
                    el = self.driver.find_element(by, sel)
                    text = el.text.strip()
                    if text:
                        return text
                except (NoSuchElementException, StaleElementReferenceException):
                    continue
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 5: Lấy tất cả button có class "CsEnBe" chứa div.Io6YTe — tìm dòng có dấu hiệu là địa chỉ
        try:
            buttons = self.driver.find_elements(By.CSS_SELECTOR, 'button.CsEnBe')
            for btn in buttons:
                try:
                    io6 = btn.find_element(By.CSS_SELECTOR, 'div.Io6YTe')
                    text = io6.text.strip()
                    addr_keywords = ['Quận', 'District', 'Phường', 'Ward', 'Đường',
                                     'Street', 'TP', 'Thành phố', 'Ho Chi Minh',
                                     'Hồ Chí Minh', 'Việt Nam', 'Vietnam']
                    if text and any(kw.lower() in text.lower() for kw in addr_keywords):
                        return text
                except NoSuchElementException:
                    continue
        except Exception:
            pass

        print(f"Không tìm thấy địa chỉ")
        return ''

    def _parse_price_text(self, text):
        # Định dạng USD
        usd_patterns = [
            r'(\d+[.,]\d+)\s*US\$',                  
            r'(\d+[.,]\d+)\s*(?:USD|\$)',             
            r'(?:US\$|USD|\$)\s*(\d+[.,]\d+)',       
            r'(\d+)\s*US\$',                          
            r'(\d+)\s*(?:USD|\$)',                     
            r'(?:US\$|USD|\$)\s*(\d+)',               
        ]
        
        # Định dạng VND
        vnd_patterns = [
            r'(\d{1,3}(?:\.\d{3})+)\s*(?:đ|đồng|VND|VNĐ|vnđ|₫)',
            r'(\d[\d\.]*)\s*(?:đ|đồng|VND|VNĐ|vnđ|₫)',
            r'(?:giá vé|ticket|admission|entrance fee|vé vào cửa)[:\s]*(\d[\d\.]*)',
        ]

        for pattern in usd_patterns:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                price_str = m.group(1).replace(',', '.')
                try:
                    usd_val = float(price_str)
                    if 0.01 <= usd_val <= 1000:
                        result = int(usd_val * 26000)
                        return result
                except ValueError:
                    continue

        for pattern in vnd_patterns:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                price_str = m.group(1).replace('.', '')
                try:
                    price_val = int(price_str)
                    if 1000 <= price_val <= 10_000_000:
                        return price_val
                except ValueError:
                    continue

        return None

    def _extract_price(self):
        # Cách 1: Tìm trong tab Tổng quan (Overview)
        ticket_xpaths = [
            '//div[contains(text(),"Vé vào cửa") or contains(text(),"Admission") or contains(text(),"Tickets")]/..',
            '//h2[contains(text(),"Vé vào cửa") or contains(text(),"Admission") or contains(text(),"Tickets")]/..',
            '//div[contains(@aria-label,"Vé vào cửa") or contains(@aria-label,"ticket") or contains(@aria-label,"Admission")]',
        ]

        for xpath in ticket_xpaths:
            try:
                section = self.driver.find_element(By.XPATH, xpath)
                text = section.text
                price = self._parse_price_text(text)
                if price:
                    return price
            except NoSuchElementException:
                continue

        # Cách 2: Tìm trong tab Vé
        try:
            ticket_tab = None
            tab_selectors = [
                (By.CSS_SELECTOR, "button[aria-label*='Vé']"),
                (By.CSS_SELECTOR, "button[aria-label*='Tickets']"),
                (By.CSS_SELECTOR, "div[role='tab'][aria-label*='Vé']"),
                (By.CSS_SELECTOR, "div[role='tab'][aria-label*='Tickets']"),
                (By.XPATH, '//button[.//div[contains(text(),"Vé")] or .//div[contains(text(),"Tickets")]]'),
                (By.XPATH, '//div[@role="tab" and (contains(.,"Vé") or contains(.,"Tickets"))]'),
                (By.XPATH, '//div[@role="tablist"]//button[contains(.,"Vé") or contains(.,"Tickets")]'),
                (By.XPATH, '//*[self::button or self::div[@role="tab"]][.//text()[normalize-space()="Vé"]]'),
            ]

            for by, sel in tab_selectors:
                try:
                    ticket_tab = self.driver.find_element(by, sel)
                    if ticket_tab.is_displayed():
                        break
                    ticket_tab = None
                except NoSuchElementException:
                    continue

            if ticket_tab:
                self.driver.execute_script("arguments[0].click();", ticket_tab)
                self._random_sleep(1, 2)

                price_found = None

                # Cách 2a: Ưu tiên giá vé của trang web chính thức
                try:
                    price_elements = self.driver.find_elements(By.XPATH,
                        '//*[contains(text(),"US$") or contains(text(),"USD") '
                        'or contains(text(),"₫") or contains(text(),"VND") '
                        'or contains(text(),"đ")]'
                    )
                    for el in price_elements:
                        el_text = el.text.strip()
                        if not el_text:
                            continue
                        price = self._parse_price_text(el_text)
                        if price:
                            price_found = price
                            break  
                except Exception:
                    pass

                # Cách 2b: Nếu không tìm được giá vé trong các element nhỏ, thử tìm trong các container lớn hơn
                if not price_found:
                    content_selectors = [
                        'div.m6QErb.DxyBCb',
                        'div.m6QErb',
                        'div[role="main"]',
                    ]
                    for sel in content_selectors:
                        try:
                            els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                            for el in els:
                                el_text = el.text
                                if not el_text:
                                    continue
                                price = self._parse_price_text(el_text)
                                if price:
                                    price_found = price
                                    break
                            if price_found:
                                break
                        except NoSuchElementException:
                            continue

                # Quay lại tab Tổng quan -> để thực hiện việc lấy các thuộc tính khác
                try:
                    overview_tab = self.driver.find_element(By.XPATH,
                        '//button[.//div[contains(text(),"Tổng quan") or contains(text(),"Overview")]] | '
                        '//div[@role="tab" and (contains(.,"Tổng quan") or contains(.,"Overview"))]')
                    self.driver.execute_script("arguments[0].click();", overview_tab)
                    self._random_sleep(0.5, 1)
                except NoSuchElementException:
                    pass

                if price_found:
                    return price_found
                else:
                    print(f"Không thu thập được giá vé")
            else:
                print(f"Không thu thập được giá vé")

        except Exception as e:
            print(f"Lỗi trong việc lấy giá vé: {e}")

        # Cách 3: Tìm trong tab Giới thiệu 
        about_selectors = [
            'div.WeS02d.fontBodyMedium',
            'div[class*="PYvSYb"]',
            'div.rogA2c',
        ]
        for sel in about_selectors:
            try:
                el = self.driver.find_element(By.CSS_SELECTOR, sel)
                price = self._parse_price_text(el.text)
                if price:
                    print(f"    [PRICE] Found in about section: {price} VNĐ")
                    return price
            except NoSuchElementException:
                continue

        return None

    def _get_comments(self, place_url, place_id):
        max_reviews = self.config['max_reviews_per_place']
        comments = []
        scores = []

        try:
            reviews_tab = self._find_reviews_tab()
            if not reviews_tab:
                return comments

            reviews_tab.click()
            self._random_sleep(2, 3)

            self._sort_reviews_newest()
            self._scroll_reviews(max_reviews)

            review_elements = self.driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf')

            for elem in review_elements[:max_reviews]:
                try:
                    try:
                        more_btn = elem.find_element(By.CSS_SELECTOR, 'button.w8nwRe.kyuRq')
                        more_btn.click()
                        time.sleep(0.3)
                    except (NoSuchElementException, ElementClickInterceptedException):
                        pass

                    try:
                        review_time = elem.find_element(
                            By.CSS_SELECTOR, 'span.rsqaWe'
                        ).text.strip()
                    except NoSuchElementException:
                        review_time = ''

                    if not self._is_within_timeframe(review_time):
                        continue
                    
                    # Lấy điểm đánh giá (số sao) của review
                    review_score = None
                    try:
                        star_el = elem.find_element(By.CSS_SELECTOR, 'span.kvMYJc')
                        aria = star_el.get_attribute('aria-label') or ''
                        m = re.search(r'(\d)', aria)
                        if m:
                            review_score = float(m.group(1))
                    except (NoSuchElementException, StaleElementReferenceException):
                        pass

                    try:
                        text = elem.find_element(
                            By.CSS_SELECTOR, 'span.wiI7pd'
                        ).text.strip()
                        text = re.sub(r'\s+', ' ', text).strip()
                        if text:
                            comments.append(text)
                            scores.append(review_score)
                    except NoSuchElementException:
                        pass

                except Exception:
                    continue

        except Exception as e:
            print(f"Lỗi trong việc lấy comments: {e}")

        return {'comments': comments, 'scores': scores}

    def _find_reviews_tab(self):
        for sel in [
            "button[aria-label*='Reviews']",
            "button[aria-label*='Bài đánh giá']",
            "button[aria-label*='đánh giá']",
        ]:
            try:
                return self.driver.find_element(By.CSS_SELECTOR, sel)
            except NoSuchElementException:
                continue
        try:
            return self.driver.find_element(By.XPATH,
                '//button[.//div[contains(text(),"Reviews") or '
                'contains(text(),"Đánh giá") or '
                'contains(text(),"Bài đánh giá")]]')
        except NoSuchElementException:
            return None

    def _sort_reviews_newest(self):
        try:
            sort_btn = self.driver.find_element(By.XPATH,
                '//button[@aria-label="Sort reviews" or '
                '@aria-label="Sắp xếp bài đánh giá" or '
                'contains(@aria-label,"Sort") or '
                'contains(@data-value,"sort")]')
            sort_btn.click()
            self._random_sleep(1, 2)

            newest = self.driver.find_element(By.XPATH,
                '//div[@role="menuitemradio" and '
                '(@data-index="1" or contains(.,"Newest") or contains(.,"Mới nhất"))]')
            newest.click()
            self._random_sleep(1, 2)
        except (NoSuchElementException, ElementClickInterceptedException):
            pass

    def _scroll_reviews(self, target):
        scrollable = None
        for sel in [
            'div.m6QErb.DxyBCb.kA9KIf.dS8AEf',
            'div.m6QErb.DxyBCb',
        ]:
            try:
                scrollable = self.driver.find_element(By.CSS_SELECTOR, sel)
                break
            except NoSuchElementException:
                continue
        if not scrollable:
            return

        last_count, no_change = 0, 0
        while no_change < 3:
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollHeight', scrollable
            )
            self._random_sleep(1, 2)
            elems = self.driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf')
            if len(elems) >= target:
                break
            if len(elems) == last_count:
                no_change += 1
            else:
                no_change = 0
            last_count = len(elems)

    def _is_within_timeframe(self, time_text):
        if not time_text:
            return True
        txt = time_text.lower()
        year_match = re.search(r'(\d+)\s*(năm|year)', txt)
        if year_match:
            return int(year_match.group(1)) <= self.config['years_back']
        if any(w in txt for w in [
            'tháng', 'tuần', 'ngày', 'giờ', 'phút',
            'month', 'week', 'day', 'hour', 'minute'
        ]):
            return True
        return True

    def _extract_nums_comments(self):
        try:
            # Kế bên thẻ để lấy 'star'
            spans = self.driver.find_elements(By.CSS_SELECTOR, "div.F7nice span")
            for span in spans:
                text = span.text.strip()
                if '(' in text and ')' in text:
                    clean_text = text.replace('.', '').replace(',', '').replace('(', '').replace(')', '')
                    if clean_text.isdigit():
                        return int(clean_text)
        except Exception:
            pass
        return None

    def _extract_hours(self):
        time_pattern = r'(\d{1,2}[:.]\d{2}(?:\s?[AP]M|\s?[SC]H)?)\s*[ \-–—đến]+\s*(\d{1,2}[:.]\d{2}(?:\s?[AP]M|\s?[SC]H)?)'
        
        # Đảm bảo đang ở tab Tổng quan
        try:
            tabs = self.driver.find_elements(By.CSS_SELECTOR, "div[role='tablist'] button")
            for tab in tabs:
                text = tab.text.strip()
                if "Tổng quan" in text or "Overview" in text:
                    if tab.get_attribute("aria-selected") != "true":
                        self.driver.execute_script("arguments[0].click();", tab)
                        self._random_sleep(1.5, 2.5) # Chờ xíu để trang load đủ nội dung
                    break
        except Exception as e:
            print(f"[Debug Hours] Lỗi khi chuyển tab Tổng quan: {e}")

        # Tìm biểu tượng đồng hồ
        hours_btn_selectors = [
            'button[data-item-id="oh"]',
            'button[aria-label*="giờ"]',
            'button[aria-label*="hours"]',
            'button[aria-label*="Đóng cửa"]',
            'button[aria-label*="Mở cửa"]',
            'button[aria-label*="Closed"]',
            'button[aria-label*="Open"]'
        ]
        
        hours_btn = None
        for sel in hours_btn_selectors:
            try:
                hours_btn = self.driver.find_element(By.CSS_SELECTOR, sel)
                if hours_btn:
                    break 
            except:
                continue 
                
        if not hours_btn:
            print("Không thu thập được giờ hoạt động")
            return ''

        # Cách 1: Lấy trực tiếp từ aria-label của nút
        try:
            full_aria = hours_btn.get_attribute('aria-label')
            if full_aria:
                for part in full_aria.split(';'):
                    if 'Thứ Hai' in part or 'Monday' in part:
                        if re.search(r'(Đóng cửa|Closed)', part, re.IGNORECASE): return "Đóng cửa"
                        if re.search(r'24\s?(giờ|hours|h)', part, re.IGNORECASE): return "00:00 - 23:59"
                        
                        m = re.search(time_pattern, part, re.IGNORECASE)
                        if m: return f"{m.group(1)} - {m.group(2)}"
        except Exception:
            pass

        # Cách 2: Click mở bảng
        try:
            if hours_btn.get_attribute("aria-expanded") != "true":
                self.driver.execute_script("arguments[0].click();", hours_btn)
                self._random_sleep(1, 2)
            
            rows = self.driver.find_elements(By.XPATH, "//table//tr")
            for row in rows:
                text = row.get_attribute('textContent')
                if not text:
                    continue
                    
                if 'Thứ Ba' in text or 'Tuesday' in text:
                    if re.search(r'(Đóng cửa|Closed)', text, re.IGNORECASE): return "Đóng cửa"
                    if re.search(r'24\s?(giờ|hours|h)', text, re.IGNORECASE): return "00:00 - 23:59"
                        
                    m = re.search(time_pattern, text, re.IGNORECASE)
                    if m: return f"{m.group(1)} - {m.group(2)}"
                    
        except Exception as e:
            print(f"Lỗi không lấy được giờ hoạt động: {e}") 

        return ''

    # Main loop
    def scrape(self, queries, pbar=None):
        for i, query in enumerate(queries):
            try:
                found = self.search_places(query)
                if not found:
                    continue

                num = self.scroll_results()
                urls = self.get_place_urls()

                for url in urls:
                    pid = self._extract_place_id(url)
                    if pid in self.seen_place_ids:
                        continue

                    # Thu thập được tất cả thuộc tính cho 1 địa điểm
                    record = self.extract_place(url, pid)
                    if record:
                        self.all_data.append(record)

                        if pbar:
                            pbar.update(1)
                            pbar.set_postfix({
                                'total': len(self.all_data),
                                'last': record['name'][:20],
                            })

                        # Khi đạt đủ checkpoint_interval địa điểm, lưu lại dữ liệu
                        if len(self.all_data) % self.config['checkpoint_interval'] == 0:
                            self._save_checkpoint()

                    self._random_sleep(
                        self.config['sleep_between_places'],
                        self.config['sleep_between_places'] + 2
                    )

            except Exception as e:
                print(f"Lỗi khi thực hiện tìm kiếm '{query}': {e}")
                continue

            self._random_sleep(
                self.config['sleep_between_queries'],
                self.config['sleep_between_queries'] + 2
            )
            
    @staticmethod
    def _format_comments(comments_list):
        if not comments_list or not isinstance(comments_list, list):
            return '[]'
        parts = ', '.join(f'{{{c}}}' for c in comments_list)

        return f'[{parts}]'

    @staticmethod
    def _format_scores(scores_list):
        if not scores_list or not isinstance(scores_list, list):
            return '[]'
        parts = '; '.join(str(s) if s is not None else '' for s in scores_list)
        return f'[{parts}]'
    
    # Lưu trữ
    def _save_checkpoint(self):
        try:
            self._save_to_csv(
                os.path.join(self.config['output_dir'], 'checkpoint.csv')
            )
        except Exception as e:
            print(f"Lỗi khi lưu checkpoint: {e}")

    def _save_to_csv(self, filepath):
        df = pd.DataFrame(self.all_data)
        if 'comments' in df.columns:
            df['comments'] = df['comments'].apply(self._format_comments)
        if 'comment_scores' in df.columns:
            df['comment_scores'] = df['comment_scores'].apply(self._format_scores)
        
        cols = [c for c in COLUMN_ORDER if c in df.columns]
        df = df[cols]
        df.to_csv(filepath, index=False, encoding='utf-8-sig')

    def save_final(self):
        output_dir = self.config['output_dir']
        os.makedirs(output_dir, exist_ok=True)

        # Định dạng .csv
        csv_path = os.path.join(output_dir, 'ggmaps.csv')
        self._save_to_csv(csv_path)

        # Định dạng .json
        json_path = os.path.join(output_dir, 'ggmaps.json')
        
        json_data = []
        for record in self.all_data:
            r = dict(record)
            r['comments'] = self._format_comments(r.get('comments', []))
            r['comment_scores'] = self._format_scores(r.get('comment_scores', []))
            json_data.append(r)
        
        ordered_data = [{k: r.get(k) for k in COLUMN_ORDER} for r in json_data]
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(ordered_data, f, ensure_ascii=False, indent=2)

        print(f"Dữ liệu đã được lưu vào: {csv_path} và {json_path}")

    def close(self):
        try:
            self.driver.quit()
        except Exception:
            pass


## 2.4.2 Quá trình thực hiện

Quá trình thu thập dữ liệu được thực hiện hoàn toàn tự động thông qua việc mô phỏng hành vi của một người dùng thật đang sử dụng Google Maps để tìm kiếm và khám phá các địa điểm tham quan tại TP. Hồ Chí Minh. Quy trình gồm 4 bước chính lặp lại cho từng câu truy vấn:
- Bước 1: Tìm kiếm từ khóa
- Bước 2: Duyệt qua danh sách kết quả
- Bước 3: Thu thập thông tin từng địa điểm
- Bước 4: Lưu trữ dữ liệu

In [19]:
scraper = GoogleMapsScraper(config=CONFIG)

queries = SEARCH_QUERIES['attraction']
estimated_total = len(queries) * CONFIG['max_places_per_query']

try:
    with tqdm(total=estimated_total, desc="Scraping") as pbar:
        scraper.scrape(queries, pbar=pbar)

    print(f"Thu thập được: {len(scraper.all_data)} điểm dữ liệu")

except KeyboardInterrupt:
    print(f"\nNgừng scraping do người dùng dừng.")
    print(f"Thu thập được: {len(scraper.all_data)} điểm dữ liệu")

    scraper._save_checkpoint()

except Exception as e:
    print(f"\nLỗi khi thực hiện scraping: {e}")

    scraper._save_checkpoint()
finally:
    scraper.close()

scraper.save_final()

Scraping:   0%|          | 0/13400 [00:00<?, ?it/s]

Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Lỗi trong việc extract_place: list indices must be integers or slices, not str
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giờ hoạt động
Không thu thập được g

# **3. Booking**


## 3.1 Giới thiệu

**Mục tiêu**: Thu thập dữ liệu ~2600 khách sạn tại các quận thuộc Thành phố Hồ Chí Minh

**Phân loại**: có nhãn category là `hotel`, bao gồm: khách sạn, resort, homestay, nhà nghỉ, căn hộ dịch vụ,...

**Phương pháp tiếp cận**: Nhóm sử dụng phương pháp **Web Scraping** thông qua giả lập trình duyệt với thư viện `Playwright` (Sync API) để thu thập dữ liệu từ Booking.com.

**Kết quả** (`Ho_Chi_Minh_Hotels.csv`):
| Cột | Kiểu dữ liệu | Mô tả |
|-----|------|-------|
| name | str | Tên khách sạn |
| address | str | Địa chỉ |
| score | float | Điểm đánh giá (0-5) |
| nums_comments | int | Tổng số lượt đánh giá |
| price | int | Giá phòng khác sạn |
| category | str | Phân loại nhóm |
| hours | str | Giờ hoạt động |
| comments | list str | Danh sách đánh giá |
| comment_scores | list float | Danh sách điểm đánh giá |
| url | str | Đường dẫn trên Booking |


## 3.2 Thư viện


In [ ]:
%pip install playwright pandas -q

from playwright.sync_api import sync_playwright
import pandas as pd
import time
import re
import os

## 3.3 Danh sách tìm kiếm


Để thu thập dữ liệu khách sạn một cách bao phủ và đa dạng tại TP. Hồ Chí Minh, nhóm xây dựng một danh sách các địa điểm tìm kiếm (search locations) được sử dụng để truy vấn trên Booking.com.

Cách thiết kế danh sách `PLACES`:
- Tìm kiếm theo từng quận/huyện tại TP. Hồ Chí Minh
- Bao gồm các quận có số (Quận 1, 3, 4, 5, 6, 7, 8)
- Bao gồm các quận/huyện khác (Phú Nhuận, Bình Thạnh, Tân Bình, Gò Vấp, Nhà Bè, Bình Tân, Hóc Môn, Củ Chi, Cần Giờ)
- Sử dụng format chuẩn: "Quan X TP.Ho Chi Minh" để đảm bảo Booking.com nhận diện chính xác


In [ ]:
PLACES = [
    "Quan 1 TP.Ho Chi Minh", 
    "Quan 3 TP.Ho Chi Minh", 
    "Quan 4 TP.Ho Chi Minh", 
    "Quan 5 TP.Ho Chi Minh", 
    "Quan 6 TP.Ho Chi Minh",
    "Quan 7 TP.Ho Chi Minh", 
    "Quan 8 TP.Ho Chi Minh", 
    "Quan Phu Nhuan TP.Ho Chi Minh", 
    "Quan Binh Thanh TP.Ho Chi Minh", 
    "Quan Tan Binh TP.Ho Chi Minh",
    "Quan Go Vap TP.Ho Chi Minh", 
    "Quan Nha Be TP.Ho Chi Minh", 
    "Quan Binh Tan TP.Ho Chi Minh", 
    "Quan Hoc Mon TP.Ho Chi Minh", 
    "Quan Cu Chi TP.Ho Chi Minh", 
    "Quan Can Gio TP.Ho Chi Minh"
]
CHECKIN_DATE = "2026-05-01"  
CHECKOUT_DATE = "2026-05-02"
MAX_COMMENTS_PER_HOTEL = 50

## 3.4 Thu thập dữ liệu


Nhóm khởi tạo một trình duyệt Chromium thực sự và điều khiển nó tự động thông qua **Playwright Sync API**, mô phỏng hoàn toàn các thao tác của người dùng thật: truy cập trang tìm kiếm, cuộn để tải thêm kết quả, nhấn vào từng khách sạn, và đọc nội dung hiển thị trên trang.

Quy trình thu thập dữ liệu được chia thành **2 giai đoạn** chính:

**Giai đoạn 1: Thu thập danh sách khách sạn từ trang tìm kiếm**
- Truy cập URL tìm kiếm, đọc số lượng khách sạn và cuộn trang để tải thêm kết quả để đạt số lượng mục tiêu.
- Trích xuất: `name`, `url`, `price` từ mỗi khách sạn

**Giai đoạn 2: Thu thập thông tin chi tiết từ từng trang khách sạn**
- Mở từng URL khách sạn trong một tab riêng biệt
- Trích xuất thông tin chi tiết:
  - **Điểm tổng thể & số lượng đánh giá**
  - **Địa chỉ**
  - **Bình luận**: Click nút "Đọc tất cả đánh giá", đợi popup hiển thị, lấy tối đa 50 comment đầu tiên từ các card, gộp positive & negative text
  - **Điểm bình luận**

**Xử lý dữ liệu trước khi lưu:**
- Loại bỏ khách sạn không có địa chỉ hợp lệ (address = 'N/A' hoặc None)
- Sắp xếp các cột theo thứ tự: place, name, address, score, num_comments, price, category, hours, comments, comment_scores, url

**Lưu trữ:**
- Lưu file CSV theo từng địa điểm: `{place}_Hotels_{count}.csv` vào folder `hochiminh_hotels/` (Backup file để tránh trường hợp ngắt kết nối trong quá trình thu thập dữ liệu)
- Lưu file CSV tổng hợp: `Ho_Chi_Minh_Hotels.csv` ở thư mục gốc


## 3.4.1 Khai báo hàm


Code thu thập dữ liệu được tổ chức thành 3 hàm chính: **Lưu trữ**, **Trích xuất chi tiết**, và **Thu thập tổng thể**

- Lưu trữ:
    - `save_to_csv()`: Lưu danh sách khách sạn theo từng địa điểm
        - Lọc bỏ khách sạn không có địa chỉ hợp lệ
        - Tạo cột category, hours (vì khách sạn mở cửa 24/7)
        - Sắp xếp các cột theo thứ tự: place, name, address, score, num_comments, price, category, hours, comments, comment_scores, url
        - Lưu vào folder `hochiminh_hotels/` với tên file `{place}_Hotels_{count}.csv`

- Trích xuất chi tiết:
    - `get_details_from_detail_page()`: Lấy thông tin chi tiết từ trang khách sạn
        - **Tổng điểm & số lượng đánh giá**
        - **Địa chỉ**
        - **Bình luận**: Quy trình 4 bước
            1. Tìm và click nút "Đọc tất cả đánh giá"
            2. Đợi popup hiển thị
            3. Nhấn qua từng trang để thu thập để số lượng comments
            4. Từ mỗi card, lấy positive & negative text, gộp lại: `comment = positive + ". " + negative`
            5. Từ mỗi card, thu thập điểm của bình luận đó

- Thu thập tổng thể:
    - `scrape_booking()`: Hàm main điều phối toàn bộ quy trình
        - Khởi tạo browser
        - Vòng lặp qua từng địa điểm trong PLACES
        - **Giai đoạn 1**: Thu thập danh sách từ searchresults.vi.html
            - Xây dựng URL
            - Đọc số lượng thực tế từ tiêu đề
            - Vòng lặp cuộn + click "Hiển thị thêm"
            - Điều kiện dừng: đạt target_count HOẶC stuck_count >= 5 (không tải được thêm)
            - Cắt danh sách: `final_hotels = hotels_elements[:target_count]`
            - Trích xuất từ mỗi card: name, url, price
        - **Giai đoạn 2**: Truy cập từng URL để lấy chi tiết
            - Gọi `get_details_from_detail_page()` cho từng URL
            - Merge dữ liệu: gán score, num_comments, address, comments, comment_scores vào dict hotel
            - Cập nhật lại `all_hotels[actual_idx]` để giữ nguyên thứ tự
        - Lưu file theo địa điểm sau mỗi địa điểm hoàn thành
        - Lưu file tổng ở cuối: lọc address hợp lệ, sắp xếp cột theo thứ tự format


In [ ]:
def save_to_csv(hotels_list, city_name):
    # Loại bỏ khách sạn không có địa chỉ
    hotels_list = [h for h in hotels_list if h.get('address') and h['address'] != 'N/A']

    if len(hotels_list) == 0:
        print(f"   [!] Không có khách sạn nào có địa chỉ hợp lệ cho {city_name}")
        return

    # Tạo folder
    folder = "hochiminh_hotels"
    os.makedirs(folder, exist_ok=True)
    
    # Tạo tên file
    filename = os.path.join(folder, f"{city_name.replace(' ', '_')}_Hotels_{len(hotels_list)}.csv")

    df = pd.DataFrame(hotels_list)
    
    # Điền giá trị rỗng cho các cột điểm hạng mục nếu không có giá trị
    score_cols = [col for col in df.columns if col.startswith('score_')]
    for col in score_cols:
        df[col] = df[col].fillna('')

    # Tạo cột 'hours': tất cả giá trị đều rỗng
    df['hours'] = ''

    # Tạo cột 'category': tất cả giá trị là 'hotel'
    df['category'] = 'hotel'
    
    # Sắp xếp các cột theo thứ tự: place, name, address, score, num_comments, price, category, hours, comments, comment_scores, url, score_*
    ordered_cols = ['place', 'name', 'address', 'score', 'num_comments', 'price', 'category', 'hours', 'comments', 'comment_scores', 'url'] + score_cols
    df = df[[col for col in ordered_cols if col in df.columns]]
    
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n[HOÀN THÀNH] Đã lưu file theo địa điểm: {filename} (Tổng: {len(hotels_list)} khách sạn)")


In [ ]:
def get_details_from_detail_page(page, url):
    result = {
        'address': 'N/A',
        'comments': [],
        'comment_scores': [],
        'score': '0',
        'review_count': '0',
        'sub_scores': {}
    }
    
    try:
        # Tải trang và chờ DOM ổn định
        page.goto(url, timeout=60000)
        page.wait_for_load_state('domcontentloaded')
        time.sleep(2)

        # --- LẤY TỔNG ĐIỂM VÀ SỐ LƯỢNG ĐÁNH GIÁ ---
        try:
            score_block = page.locator('[data-testid="review-score-component"]').first
            if score_block.is_visible(timeout=5000):
                full_text = score_block.inner_text() # "Đạt điểm 8.7\n8,7\nĐược đánh giá tuyệt vời\nTuyệt vời · 115 đánh giá"

                # Regex lấy điểm số
                score_match = re.search(r'(\d+(?:[.,]\d+)?)', full_text)
                if score_match:
                    score = float(score_match.group(1)) / 2 # Lấy thang 5
                    result['score'] = f"{score:.1f}"
                
                # Regex lấy số lượng đánh giá
                count_match = re.search(r'([\d.,]+)\s+đánh giá', full_text)
                if count_match:
                    raw_count = count_match.group(1)
                    result['review_count'] = re.sub(r'[^\d]', '', raw_count)
        except Exception as e:
            print(f"      [!] Lỗi trích xuất điểm số hoặc số lượng đánh giá: {e}")
        
        # --- LẤY ĐỊA CHỈ ---
        try:
            # Selector
            address_locators = [
                page.locator('[data-testid="PropertyHeaderAddressDesktop-wrapper"]').first,
                page.locator('[data-capla-component-boundary="b-property-web-property-page/PropertyHeaderAddressDesktop"]').first,
                page.locator('[data-testid="address"]').first,
            ]
            
            for loc in address_locators:
                if loc.is_visible(timeout=3000):
                    raw_text = loc.inner_text()
                    
                    clean_address = raw_text.split('\n')[0].strip()

                    if clean_address:
                        result['address'] = clean_address
                        break
        except Exception as e:
            print(f"      [!] Lỗi trích xuất địa chỉ: {e}")

        # --- LẤY ĐIỂM CÁC HẠNG MỤC (NHÂN VIÊN, SẠCH SẼ, ...) ---
        try:
            sub_elements = page.locator('[data-testid="review-subscore"]').all()

            for el in sub_elements:
                raw_text = el.inner_text() # "Nhân viên phục vụ\n9,5"
                parts = [p.strip() for p in raw_text.split('\n') if p.strip()]

                # Lưu vào dict: key = tên hạng mục, value = điểm số
                if len(parts) >= 2:
                    cat_name = parts[0]
                    cat_score = parts[-1]
                    result['sub_scores'][cat_name] = cat_score
        except Exception as e:
            print(f"      [!] Lỗi lấy điểm hạng mục: {e}")

        # --- LẤY COMMENT ---
        try:
            review_clicked = False

            # Locator dự phòng cho nút 
            btn_review = page.locator('[data-testid="review-score-read-all"], [data-testid="read-all-actionable"]').first
            review_clicked = False

            for _ in range(2):
                if btn_review.is_visible(timeout=5000):
                    btn_review.click(force=True)
                    review_clicked = True
                    break
            
            if review_clicked:
                page.wait_for_selector('[data-testid="review-card"], .c-review-block', timeout=10000)
                time.sleep(2)

                # Cuộn xuống trong phạm vi popup (không cuộn trang chính)
                def scroll_popup(to_top=False):
                    try:
                        popup = page.locator('[role="dialog"], [data-testid="review-list-page-container"], .bui-modal__body, [data-testid="reviews-container"]').first
                        if popup.is_visible(timeout=2000):
                            popup.evaluate(f"el => el.scrollTop = {'0' if to_top else 'el.scrollHeight'}")
                    except: pass

                scroll_popup()
                time.sleep(1)

                current_page_num = 1

                # Lặp qua từng trang đánh giá cho đến khi đủ 50 comment hoặc hết nút
                while len(result['comments']) < MAX_COMMENTS_PER_HOTEL:
                    # Thu thập tất cả cards trên trang hiện tại
                    review_cards = page.locator('[data-testid="review-card"], .c-review-block').all()

                    for card in review_cards:
                        if len(result['comments']) >= MAX_COMMENTS_PER_HOTEL:
                            break

                        # === LẤY ĐIỂM COMMENT ===
                        try:
                            score_loc = card.locator('[data-testid="review-score"] > div, .bui-review-score__badge').first
                            if score_loc.is_visible():
                                raw_score = score_loc.inner_text().strip()
                                clean_score = raw_score.split('\n')[-1].strip()
                                result['comment_scores'].append(clean_score)
                        except: pass

                        # === NỘI DUNG COMMENTS ===    
                        # Cấu trúc comment: String thô => "Positive. Negative"
                        comment = ""

                        # Lấy positive và negative nếu có, gộp vào 1 string
                        try:
                            pos_loc = card.locator('[data-testid="review-positive-text"], .c-review__body--positive').first
                            if pos_loc.is_visible():
                                comment += pos_loc.inner_text().strip()
                        except: pass

                        try:
                            neg_loc = card.locator('[data-testid="review-negative-text"], .c-review__body--negative').first
                            if neg_loc.is_visible():
                                comment += ". " + neg_loc.inner_text().strip()
                        except: pass

                        # Nếu comment vẫn rỗng, thử lấy text chung
                        if not comment.strip():
                            try:
                                text_loc = card.locator('[data-testid="review-text"], .c-review__body').first
                                if text_loc.is_visible():
                                    comment = text_loc.inner_text().strip()
                            except: pass

                        # Lưu comment có nội dung vào mảng
                        if comment.strip():
                            result['comments'].append(comment.strip())

                    if len(result['comments']) >= MAX_COMMENTS_PER_HOTEL:
                        break

                    # Tìm và click nút số trang tiếp theo
                    # Dùng regex exact match để tránh nhầm "12" khi tìm "1"
                    next_page_num = current_page_num + 1
                    next_btn_clicked = False

                    try:
                        next_btn = page.locator('button').filter(
                            has_text=re.compile(r'^\s*' + str(next_page_num) + r'\s*$')
                        ).first

                        if next_btn.is_visible(timeout=3000):
                            next_btn.scroll_into_view_if_needed()
                            next_btn.click(force=True)
                            time.sleep(2)
                            page.wait_for_selector('[data-testid="review-card"], .c-review-block', timeout=10000)
                            scroll_popup(to_top=True)
                            current_page_num = next_page_num
                            next_btn_clicked = True
                    except: pass

                    if not next_btn_clicked:
                        print(f"      -> Không còn trang tiếp theo (dừng ở trang {current_page_num}).")
                        break

                print(f"      -> Đã lấy {len(result['comments'])} comment.")
            else:
                print("      [!] Nút mở đánh giá không hiển thị hoặc không tồn tại.")
        except Exception as e:
            print(f"      [!] Lỗi khi mở hoặc đọc popup comment: {e}")

    except Exception as e:
        print(f"      [!] Lỗi không thể tải link chi tiết: {e}")
        
    return result

In [ ]:
def scrape_booking():
    all_hotels = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False, args=["--start-maximized"])
        context = browser.new_context(no_viewport=True)
        page = context.new_page()

        for place in PLACES:
            print(f"\n========== ĐANG QUÉT: {place} ==========")
            
            # Đánh dấu vị trí bắt đầu của thành phố này
            place_start_index = len(all_hotels)
            
            place_query = place.replace(" ", "+")
            url = f"https://www.booking.com/searchresults.vi.html?ss={place_query}&checkin={CHECKIN_DATE}&checkout={CHECKOUT_DATE}&group_adults=2&no_rooms=1"
            
            try:
                page.goto(url, timeout=60000)
                page.wait_for_selector('[data-testid="property-card"]', timeout=30000)
            except:
                print("   -> Lỗi load trang. Bỏ qua.")
                continue

            # --- TÍNH TOÁN CON SỐ MỤC TIÊU ---
            total_found = 0
            target_count = 0
            
            try:
                # Tìm thẻ H1 chứa dòng chữ "Đã tìm thấy..."
                header_text = page.locator("h1").inner_text()
                print(f"   -> Tiêu đề trên web: {header_text}")
                
                # Regex tìm số lượng sau "tìm thấy" (e.g. "1.037" hoặc "56")
                match = re.search(r'tìm thấy\s+([\d.,]+)', header_text, re.IGNORECASE)
                
                if match:
                    total_found = int(re.sub(r'[^\d]', '', match.group(1)))
                    print(f"   => Tổng số khách sạn thực tế: {total_found}")
                    target_count = total_found
                        
                    print(f"   => MỤC TIÊU CẦN LẤY: {target_count} khách sạn.")
                else:
                    print("   => Không đọc được số lượng. Chạy mặc định lấy 50 cái.")
                    target_count = 50
            except:
                target_count = 50   

            # --- VÒNG LẶP CUỘN CHO ĐẾN KHI ĐỦ SỐ LƯỢNG MỤC TIÊU ---
            last_count = 0
            stuck_count = 0
            
            while True:
                current_cards = page.locator('[data-testid="property-card"]').count()
                print(f"   [Loading] Đang có {current_cards}/{target_count}...")

                if current_cards >= target_count:
                    print("   => Đã tải đủ số lượng yêu cầu.")
                    break
                
                # Hết dữ liệu
                if current_cards == last_count:
                    stuck_count += 1
                    if stuck_count >= 5: # Thử 5 lần cuộn
                        print("   => Web không còn gì để tải thêm (Đã hết danh sách).")
                        break
                else:
                    stuck_count = 0
                    last_count = current_cards

                # Tắt pop-up (nếu có)
                try:
                    close_btn = page.locator('[aria-label="Đóng"], [aria-label="Close"], [data-testid="modal-mask"], button.modal-mask-closeBtn').first
                    if close_btn.is_visible(timeout=4000):
                        close_btn.click(force=True)
                    else:
                        page.keyboard.press("Escape")
                except: pass

                # Cuộn và Click nút "Hiển thị thêm" nếu có
                page.keyboard.press("End")
                time.sleep(1.5)
                
                try:
                    btn = page.locator("button, a").filter(has_text="Hiển thị thêm").or_(
                          page.locator("button, a").filter(has_text="Tải thêm kết quả")).or_(
                          page.locator("button, a").filter(has_text="Load more")).first
                    if btn.is_visible():
                        btn.click(force=True)
                        time.sleep(2)
                except: pass

            # --- CẮT DỮ LIỆU ĐÚNG SỐ LƯỢNG ---
            print(f"   -> Đang xử lý dữ liệu...")
            hotels_elements = page.locator('[data-testid="property-card"]').all()
            
            # Cắt danh sách: Chỉ lấy từ 0 đến target_count
            final_hotels = hotels_elements[:target_count]
            
            print(f"   -> Đã cắt danh sách. Số lượng chốt sổ: {len(final_hotels)}")

            count = 0
            for hotel in final_hotels:
                item = {}
                item['place'] = place
                try:
                    # --- LẤY THÔNG TIN CƠ BẢN ---
                    item['name'] = hotel.locator('[data-testid="title"]').inner_text()
                    item['url'] = hotel.locator('a[data-testid="title-link"]').get_attribute('href')
                    
                    try:
                        price = hotel.locator('[data-testid="price-and-discounted-price"]').inner_text().replace("VND ", "")
                        price = re.sub(r'[^\d]', '', price) # Chỉ giữ lại số
                        item['price'] = price
                    except:
                        item['price'] = "N/A"

                    all_hotels.append(item)

                    print(f"   [{count+1}/{len(final_hotels)}] Đã lấy: {item['name']} - Price: {item['price']}")
                    count += 1
                except: continue

            # --- GIAI ĐOẠN 2: TRUY CẬP TỪNG LINK ĐỂ LẤY COMMENT & ADDRESS ---
            current_place_hotels = all_hotels[place_start_index:]
            
            print(f"\n========== BẮT ĐẦU VÀO TRANG CHI TIẾT LẤY COMMENT & ĐỊA CHỈ CHO {place} ==========")
            page_detail = context.new_page()

            for idx, hotel in enumerate(current_place_hotels):
                actual_idx = place_start_index + idx
                print(f"  [{idx+1}/{len(current_place_hotels)}] Đang quét chi tiết: {hotel['name']}...")
                
                # Gọi hàm xử lý trang chi tiết
                detail_data = get_details_from_detail_page(page_detail, hotel['url'])

                # Gán điểm tổng thể và số lượng đánh giá
                hotel['score'] = detail_data['score']
                hotel['num_comments'] = detail_data['review_count']
                
                # Gán Address
                if detail_data['address'] != 'N/A':
                    hotel['address'] = detail_data['address']
                    
                # Gán điểm hạng mục (nếu có)
                for cat_name, cat_score in detail_data['sub_scores'].items():
                    hotel[f'score_{cat_name}'] = cat_score
                
                # Gán comment
                hotel['comments'] = detail_data['comments']

                # Gán điểm comment (parse sang format string: [điểm 1; điểm 2; ...])
                if detail_data.get('comment_scores'):
                    hotel['comment_scores'] = f"[{'; '.join(detail_data['comment_scores'])}]"
                else:
                    hotel['comment_scores'] = "[]"
                
                print(f"      -> Điểm tổng thể: {hotel['score']} (Dựa trên {hotel['num_comments']} đánh giá)")
                print(f"      -> Địa chỉ: {hotel.get('address')}")
                print(f"      -> Hạng mục: {detail_data['sub_scores']}")
                print(f"      -> Lấy được {len(detail_data['comments'])} comment.")
                all_hotels[actual_idx] = hotel
                time.sleep(1.5)

            page_detail.close()
            
            # --- LƯU FILE TẠM THEO ĐỊA ĐIỂM ---
            save_to_csv(current_place_hotels, place)

    # --- LƯU FILE TỔNG ---
    if len(all_hotels) > 0:
        df = pd.DataFrame(all_hotels)
        
        # Loại bỏ khách sạn không có địa chỉ
        df = df[df['address'].notna() & (df['address'] != 'N/A')]
        
        # Điền giá trị rỗng cho các cột điểm hạng mục nếu không có giá trị
        score_cols = [col for col in df.columns if col.startswith('score_')]
        for col in score_cols:
            df[col] = df[col].fillna('')

        # Tạo cột 'hours': tất cả giá trị đều rỗng
        df['hours'] = ''

        # Tạo cột 'category': tất cả giá trị là 'hotel'
        df['category'] = 'hotel'
        
        # Sắp xếp các cột theo thứ tự: place, name, address, score, num_comments, price, category, hours, comments, comment_scores, url, score_*
        ordered_cols = ['place', 'name', 'address', 'score', 'num_comments', 'price', 'category', 'hours', 'comments', 'comment_scores', 'url'] + score_cols
        df = df[[col for col in ordered_cols if col in df.columns]]
        
        df.to_csv(f'Ho_Chi_Minh_Hotels_{len(df)}.csv', index=False, encoding='utf-8-sig')
        print(f"\n[HOÀN THÀNH] Đã lưu file tổng: Ho_Chi_Minh_Hotels_{len(df)}.csv (Tổng: {len(df)} khách sạn)")
    else:
        print("Lỗi: Không lấy được dữ liệu.")

## 3.4.2 Quá trình thực hiện


Quá trình thu thập dữ liệu được thực hiện hoàn toàn tự động thông qua việc mô phỏng hành vi của một người dùng thật đang sử dụng Booking.com để tìm kiếm và khám phá các khách sạn tại TP. Hồ Chí Minh. Quy trình gồm 2 giai đoạn chính được lặp lại cho từng địa điểm:

**Giai đoạn 1: Thu thập danh sách từ trang tìm kiếm**
- Bước 1.1: Xây dựng URL tìm kiếm với query string
- Bước 1.2: Tải trang và chờ property cards xuất hiện
- Bước 1.3: Đọc tổng số khách sạn từ tiêu đề
- Bước 1.4: Vòng lặp cuộn để tải thêm kết quả
  - Điều kiện dừng:
    - `current_cards >= target_count`: đủ số lượng
    - `stuck_count >= 5`: không tải được thêm sau 5 lần thử
- Bước 1.5: Cắt danh sách chính xác
  - Đảm bảo số lượng chính xác, không dư thừa
- Bước 1.6: Trích xuất thông tin cơ bản từ mỗi property card: `name`, `url`, `price`

**Giai đoạn 2: Thu thập chi tiết từ từng trang khách sạn**
- Bước 2.1: Mở tab mới để xử lý song song
- Bước 2.2: Duyệt qua từng URL trong danh sách của địa điểm hiện tại
- Bước 2.3: Trong `get_details_from_detail_page()`:
  - Trích xuất điểm & số lượng đánh giá.
  - Trích xuất địa chỉ.
  - Trích xuất bình luận (nội dung và điểm của bình luận)
- Bước 2.4: Merge dữ liệu vào dict hotel
- Bước 2.5: Đóng tab chi tiết sau khi hoàn thành địa điểm

**Lưu trữ dữ liệu:**
- Lưu file theo địa điểm ngay sau khi hoàn thành mỗi địa điểm
  - Gọi `save_to_csv(current_place_hotels, place)`
  - Tên file: `{place}_Hotels_{count}.csv` trong folder `hochiminh_hotels/`
- Lưu file tổng sau khi hoàn thành tất cả: `Ho_Chi_Minh_Hotels.csv`


In [ ]:
if __name__ == '__main__':
    scrape_booking()


# **4. Foody**


## 4.1 Giới thiệu

**Mục tiêu**: Thu thập dữ liệu ~1800 nhà hàng/quán ăn tại 24 quận/huyện thuộc Thành phố Hồ Chí Minh

**Phân loại**: có nhãn category là `food`, bao gồm: nhà hàng, quán ăn, quán cafe, quán nhậu, chuỗi thương hiệu ẩm thực,...

**Phương pháp tiếp cận**: Nhóm sử dụng phương pháp **Web Scraping** thông qua giả lập trình duyệt với thư viện `Playwright` (Sync API) để thu thập dữ liệu từ Foody.vn.

**Kết quả** (`tphcm_food.csv`):
| Cột | Kiểu dữ liệu | Mô tả |
|-----|------|-------|
| name | str | Tên địa điểm |
| address | str | Địa chỉ |
| score | float | Điểm đánh giá (0-5) |
| nums_comments | int | Tổng số lượt đánh giá |
| price | int | Giá món ăn và đồ uống |
| category | str | Phân loại nhóm |
| hours | str | Giờ hoạt động |
| comments | list str | Danh sách đánh giá |
| comment_scores | list float | Danh sách điểm đánh giá |
| url | str | Đường dẫn trên Foody |

## 4.2 Thư viện


In [ ]:
%pip install playwright pandas -q

from playwright.sync_api import sync_playwright
import pandas as pd
import time
import re
import os

## 4.3 Danh sách tìm kiếm


Để thu thập dữ liệu nhà hàng/quán ăn một cách toàn diện và bao phủ tại TP. Hồ Chí Minh, nhóm xây dựng một danh sách 24 slug địa điểm (district slugs) được sử dụng để truy vấn trên Foody.vn.

Cách thiết kế danh sách `DISTRICTS`:
- Tìm kiếm theo từng quận/huyện tại TP. Hồ Chí Minh
- Bao gồm 12 quận có số (Quận 1 đến Quận 12)
- Bao gồm các quận khác: Bình Thạnh, Tân Bình, Phú Nhuận, Tân Phú, Gò Vấp, Bình Tân
- Bao gồm Thủ Đức (thành phố trực thuộc)
- Bao gồm 5 huyện: Bình Chánh, Nhà Bè, Hóc Môn, Củ Chi, Cần Giờ
- Sử dụng format slug chuẩn của Foody: "quan-1", "quan-binh-thanh", "huyen-cu-chi",... để đảm bảo URL hợp lệ


In [ ]:
DISTRICTS = [
    "quan-1", "quan-2", "quan-3", "quan-4", "quan-5", "quan-6", 
    "quan-7", "quan-8", "quan-9", "quan-10", "quan-11", "quan-12",
    "quan-binh-thanh", "quan-tan-binh", "quan-phu-nhuan", "quan-tan-phu", 
    "quan-go-vap", "quan-binh-tan", "thu-duc", "huyen-binh-chanh", 
    "huyen-nha-be", "huyen-hoc-mon", "huyen-cu-chi", "huyen-can-gio"
]
MAX_COMMENTS_PER_RESTAURANT = 50

## 4.4 Thu thập dữ liệu


Nhóm khởi tạo một trình duyệt Chromium thực sự và điều khiển nó tự động thông qua **Playwright Sync API**, mô phỏng hoàn toàn các thao tác của người dùng thật: truy cập trang danh sách quán ăn, cuộn để tải thêm kết quả, phát hiện chuỗi thương hiệu, và đọc nội dung hiển thị trên trang.

Quy trình thu thập dữ liệu được chia thành **4 giai đoạn** chính:

**Giai đoạn 1: Thu thập danh sách quán ăn từ trang tìm kiếm theo quận**
- Truy cập URL: `https://www.foody.vn/ho-chi-minh/dia-diem-tai-{district}?categorygroup=food`
- Vòng lặp cuộn và click "Xem thêm" để tải toàn bộ quán trong quận (Dừng khi `stuck_count >= 5` (không tải thêm được sau 5 lần thử) HOẶC gặp popup bắt đăng nhập)
- Trích xuất thông tin cơ bản: `name`, `url`, `address`, `score`

**Giai đoạn 2: Phân loại quán lẻ và chuỗi thương hiệu**
- Duyệt qua từng item trong danh sách của quận
- Phát hiện chuỗi thương hiệu qua 2 dấu hiệu:
  - `address == "Nhiều chi nhánh"`
  - HOẶC URL chứa `/thuong-hieu/`
- Nếu là quán lẻ: xử lý trực tiếp (giai đoạn 4)
- Nếu là chuỗi: chuyển sang giai đoạn 3

**Giai đoạn 3: Lấy danh sách chi nhánh từ trang thương hiệu**
- Gọi hàm `get_branches_from_chain_page(page, url, parent_item)`
- Truy cập URL chuỗi thương hiệu
- Vòng lặp click "Xem thêm" để hiển thị toàn bộ chi nhánh. (Dừng khi đã đếm hết HOẶC không tải thêm được sau 3 lần thử - `stuck_count >= 3`)
- Trích xuất từ mỗi item: `name`, `url`, `address`, và kế thừa `district` và `score` từ parent_item
- Trả về mảng các chi nhánh

**Giai đoạn 4: Thu thập thông tin chi tiết từ từng trang quán**
- Gọi hàm `get_details_from_detail_page(page, url)` cho mỗi quán/chi nhánh
- **Bộ lọc lắng nghe API**: Đăng ký listener lắng nghe XHR/Fetch response chứa review/comment → tự động trích xuất `Description` và `AvgRating`
- **Kiểm tra quán còn hoạt động**:
  - Tìm element `.micro-notification` chứa text "chưa có tương tác"
  - Nếu tìm thấy → set `is_active = False` và return ngay
- **Trích xuất thông tin chi tiết**:
  - Lấy điểm tổng thể (normalize từ thang 10 về thang 5)
  - Lấy khoảng giá
    - Nếu `score == '0'` VÀ `price == '0'` → bỏ qua quán
  - Lấy điểm hạng mục (`sub_scores`)
  - Lấy giờ mở cửa (`hours`)
  - Lấy số lượng bình luận (`num_comments`)
  - Lấy danh sách bình luận kèm điểm bình luận (`comment_scores`):
    - Ưu tiên dùng dữ liệu API đã intercept, nếu không có → fallback về DOM tĩnh

**Xử lý dữ liệu trước khi lưu:**
- Loại bỏ quán không có địa chỉ hợp lệ (address = 'N/A' hoặc None)
- Sắp xếp các cột theo thứ tự: district, name, address, score, num_comments, price, category, hours, comments, comment_scores, url

**Xử lý vấn đề đặc thù của Foody:**
- **Login Popup**: Nhấn phím `Escape` trước mỗi lần cuộn để đóng popup đăng nhập
- **Lazy Loading**: Cuộn trang theo tỷ lệ để kích hoạt lazy-load comments
- **Phát hiện quán ngưng hoạt động**: Kiểm tra notification warning và bỏ qua

**Lưu trữ:**
- Lưu file CSV theo từng quận ngay sau khi hoàn thành: `{district}_Restaurants_{count}.csv` vào folder `tphcm_foody_data/`  (Backup file để tránh trường hợp ngắt kết nối trong quá trình thu thập dữ liệu)
- Lưu file CSV tổng hợp sau khi hoàn thành 24 quận:
  - **Khử trùng lặp**: vì Foody đôi khi hiển thị 1 quán nhiều lần
  - Sắp xếp các cột theo thứ tự: district, name, address, score, num_comments, price, category, hours, comments, comment_scores, url
  - Lưu: `tphcm_food.csv`


## 4.4.1 Khai báo hàm


Code thu thập dữ liệu được tổ chức thành 4 hàm chính: **Lưu trữ**, **Trích xuất chi tiết**, **Lấy chi nhánh chuỗi**, và **Thu thập tổng thể**

- Lưu trữ:
    - `save_to_csv()`: Lưu danh sách quán ăn theo từng quận
        - Lọc bỏ quán không có địa chỉ hợp lệ
        - Sắp xếp lại cột: đưa comments xuống cuối
        - Lưu vào folder `tphcm_foody_data/` với tên file `{district}_Restaurants_{count}.csv`

- Trích xuất chi tiết:
    - `get_details_from_detail_page()`: Lấy thông tin chi tiết từ trang quán ăn
        - **Bộ lọc lắng nghe API**: Lắng nghe XHR/Fetch response chứa review/comment, tự động trích xuất `Description` và `AvgRating` (normalize về thang 5)
        - **Kiểm tra trạng thái hoạt động**
        - **Điểm tổng thể** (normalize từ thang 10 về thang 5)
        - **Khoảng giá**
        - **Kiểm tra dữ liệu hợp lệ**: Nếu score = '0' VÀ price = '0' → quán mới/chưa đủ data → bỏ qua
        - **Điểm hạng mục** (`sub_scores`)
        - **Giờ mở cửa** (`hours`)
        - **Số lượng bình luận** (`num_comments`):
        - **Bình luận + điểm bình luận** (`comment_scores`): Quy trình 4 bước
            1. Cuộn trang theo tỷ lệ 20 lần để kích hoạt lazy-load
            2. Kiểm tra có bình luận (timeout 3s)
            3. Click "Xem thêm" tối đa 3 lần
            4. Ưu tiên dữ liệu từ API đã intercept, fallback về DOM tĩnh

- Lấy chi nhánh chuỗi:
    - `get_branches_from_chain_page()`: Lấy danh sách chi nhánh từ trang thương hiệu
        - Input: URL chuỗi, parent_item (chứa district và score)
        - Tải trang thương hiệu
        - Vòng lặp click "Xem thêm" để hiển thị toàn bộ chi nhánh
        - Trích xuất từ mỗi item: `name`, `url`, `address`, và kế thừa `district` và `score` từ parent_item
        - Return: mảng các chi nhánh

- Thu thập tổng thể:
    - `scrape_food()`: Hàm main điều phối toàn bộ quy trình
        - Khởi tạo browser
        - Vòng lặp qua 24 quận trong DISTRICTS
        - **Giai đoạn 1**: Thu thập danh sách
            - Vòng lặp scroll + click load more (Dừng khi stuck_count >= 5 HOẶC không còn nút load more HOẶC xuất hiện `.login-popup`)
            - Trích xuất từ mỗi item: url, name, address, score
            - Phát hiện "Nhiều chi nhánh" nếu có `.branch-address`
        - **Giai đoạn 2-3**: Phân loại và xử lý chuỗi thương hiệu
            - Kiểm tra: `address == "Nhiều chi nhánh"` HOẶC `/thuong-hieu/` trong URL
            - Nếu là chuỗi: gọi `get_branches_from_chain_page()` → nhận mảng chi nhánh
            - Nếu là quán lẻ: xử lý trực tiếp item đó
        - **Giai đoạn 4**: Thu thập chi tiết
            - Duyệt qua từng item (có thể là 1 quán lẻ hoặc nhiều chi nhánh)
            - Gọi `get_details_from_detail_page()` cho mỗi URL
            - Kiểm tra `is_active`: nếu False → bỏ qua
            - Merge dữ liệu: price, sub_scores, score, hours, num_comments, comment_scores, comments
            - Giới hạn comments: `[:MAX_COMMENTS_PER_RESTAURANT]`
            - Thêm vào `valid_district_restaurants` và `all_restaurants`
        - Lưu file theo quận sau mỗi quận hoàn thành
        - **Lưu file tổng**: 
            - Khử quán trùng lặp
            - Đổi thứ tự cột (đẩy comments xuống cuối)
            - Lưu: `tphcm_food.csv`


In [ ]:
def save_to_csv(restaurants_list, city_name):
    # Loại bỏ quán không có địa chỉ
    restaurants_list = [h for h in restaurants_list if h.get('address') and h['address'] != 'N/A']

    if len(restaurants_list) == 0:
        print(f"   [!] Không có quán ăn nào hợp lệ để lưu cho {city_name}")
        return

    # Tạo folder
    folder = "tphcm_food"
    os.makedirs(folder, exist_ok=True)
    
    # Tạo tên file
    filename = os.path.join(folder, f"{city_name.replace('-', '_')}_Restaurants_{len(restaurants_list)}.csv")

    df = pd.DataFrame(restaurants_list)
    
    # Điền giá trị rỗng vào điểm khuyết thiếu
    score_cols = [col for col in df.columns if col.startswith('score_')]
    for col in score_cols:
        df[col] = df[col].fillna('')

    # Tạo cột 'category': tất cả giá trị là 'dining'
    df['category'] = 'dining'

    # Sắp xếp các cột theo thứ tự: district, name, address, score, num_comments, price, category, hours, comments, comment_scores, url, score_*
    ordered_cols = ['district', 'name', 'address', 'score', 'num_comments', 'price', 'category', 'hours', 'comments', 'comment_scores', 'url'] + score_cols
    df = df[ordered_cols]
    
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n[HOÀN THÀNH] Đã lưu file theo quận: {filename} (Tổng: {len(restaurants_list)} quán)")


In [ ]:
def get_details_from_detail_page(page, url):
    result = {
        'is_active': True, 
        'score': 'N/A',
        'price': 'N/A',
        'sub_scores': {},
        'comments': [],
        'num_comments': '0',
        'comment_scores': [],
        'hours': '-'
    }
    
    # ======= BỘ LỌC LẮNG NGHE API =======
    intercepted_comments = []
    intercepted_scores = []

    def intercept_api_response(response):
        if response.request.resource_type in ["xhr", "fetch"]:
            url_lower = response.url.lower()
            if "review" not in url_lower and "comment" not in url_lower:
                return

            target_path = url.split('.vn')[-1].split('?')[0].lower()
            req_url = response.request.url.lower()
            referer = response.request.headers.get('referer', '').lower()
            
            if target_path not in req_url and target_path not in referer:
                return  

            try:
                if "application/json" in response.headers.get("content-type", ""):
                    json_data = response.json()
                    
                    items = []
                    if isinstance(json_data, dict):
                        items = json_data.get('Items', [])
                        if not items and 'd' in json_data and isinstance(json_data['d'], list):
                            items = json_data['d']
                    elif isinstance(json_data, list):
                        items = json_data

                    found_comments_in_this_api = 0
                    
                    for item in items:
                        if isinstance(item, dict):
                            if 'Description' in item and 'AvgRating' in item:
                                found_comments_in_this_api += 1
                                
                                raw_desc = item.get('Description') if item.get('Description') is not None else item.get('description')
                                raw_rating = item.get('AvgRating') if item.get('AvgRating') is not None else item.get('avgRating')
                                
                                desc = str(raw_desc).strip() if raw_desc is not None else "N/A"
                                if desc == "": desc = "N/A"
                                
                                intercepted_comments.append(desc)
                                
                                if raw_rating is not None:
                                    try:
                                        normalized = float(raw_rating) / 2
                                        if normalized.is_integer():
                                            intercepted_scores.append(str(int(normalized)))
                                        else:
                                            intercepted_scores.append(str(normalized).replace('.', ','))
                                    except:
                                        intercepted_scores.append(str(raw_rating))
                                else:
                                    intercepted_scores.append("N/A")
                    
            except Exception as e:
                pass

    try:
        page.on("response", intercept_api_response)
        
        page.goto(url, timeout=60000)
        page.wait_for_load_state('domcontentloaded')
        time.sleep(1.5)

        # --- KIỂM TRA QUÁN CÒN HOẠT ĐỘNG HAY KHÔNG ---
        try:
            warning = page.locator('.micro-notification', has_text="chưa có tương tác")
            if warning.count() > 0 and warning.first.is_visible(timeout=2000):
                print("      [!] Quán này đã ngưng hoạt động hoặc không còn tương tác. Bỏ qua!")
                result['is_active'] = False
                return result
        except: pass

        # --- LẤY ĐIỂM TỔNG CỦA QUÁN (Thang 5) ---
        try:
            overall_score_loc = page.locator('.microsite-point-avg, [itemprop="aggregateRating"] [itemprop="ratingValue"]').first
            if overall_score_loc.count() > 0:
                raw_score = overall_score_loc.inner_text(timeout=2000).strip()
                try:
                    float_score = float(raw_score.replace(',', '.'))
                    normalized = float_score / 2
                    # Làm tròn đến 1 chữ số thập phân nếu cần
                    normalized = round(normalized, 1)
                    result['score'] = str(int(normalized)) if normalized.is_integer() else str(normalized)
                except:
                    result['score'] = '0'
        except Exception as e:
            print(f"      [!] Lỗi lấy điểm tổng: {e}")

        # --- LẤY KHOẢNG GIÁ (PRICE) ---
        try:
            price_loc = page.locator('.res-common-minmaxprice [itemprop="priceRange"]')
            if price_loc.count() > 0:
                raw_price = price_loc.first.inner_text().strip() 
                clean_string = raw_price.replace('.', '').replace('đ', '').replace(',', '')
                match = re.search(r'(\d+\s*-\s*\d+)', clean_string)
                if match:
                    result['price'] = match.group(1).replace(' ', '')
                else:
                    nums = re.findall(r'\d+', clean_string)
                    result['price'] = " - ".join(nums) if nums else '0'
        except Exception as e:
            print(f"      [!] Không lấy được giá: {e}")

        # NẾU KHÔNG CÓ ĐIỂM VÀ GIÁ => BỎ QUA
        if result['score'] == '0' and result['price'] == '0':
            print("      [!] Quán này không có điểm và giá, có thể là quán mới hoặc chưa đủ dữ liệu. Bỏ qua!")
            result['is_active'] = False
            return result

        # --- LẤY ĐIỂM CÁC HẠNG MỤC ---
        try:
            score_groups = page.locator('.microsite-top-points').all()
            for sg in score_groups:
                try:
                    score_val = sg.locator('div').first.inner_text(timeout=1000).strip()
                    score_label = sg.locator('.label').inner_text(timeout=1000).strip()
                    if score_label and score_val:
                        result['sub_scores'][score_label] = score_val
                except:
                    continue
        except Exception as e:
            print(f"      [!] Lỗi lấy điểm hạng mục: {e}")

        # --- LẤY GIỜ MỞ CỬA ---
        try:
            hours_loc = page.locator(
                '.micro-timesopen .itsopen + span, '
                '.micro-timesopen .itsclosed + span, '
                '.micro-timesopen > span:nth-child(3), '
                '.micro-timesopen > span:not([class])'
            ).first
            hours_text = hours_loc.inner_text(timeout=2000).strip()
            
            if hours_text:
                result['hours'] = hours_text
        except Exception as e:
            print(f"      [!] Lỗi lấy giờ mở cửa: {e}")

        # --- LÂY SỐ LƯỢNG BÌNH LUẬN ---
        try:
            comment_count_loc = page.locator('.microsite-review-count').first
            comment_text = comment_count_loc.inner_text(timeout=2000).strip()
            if comment_text:
                match = re.search(r'(\d+)', comment_text.replace('.', '').replace(',', ''))
                if match:
                    result['num_comments'] = match.group(1)
        except Exception as e:
            print(f"      [!] Lỗi lấy số lượng bình luận: {e}")

        # --- CUỘN VÀ LẤY COMMENT ---
        try:
            for i in range(1, 20):
                page.evaluate(f"window.scrollTo(0, document.body.scrollHeight * ({i}/10));")
                time.sleep(0.5)
            
            has_comments = False
            try:
                page.locator('.review-des, .rd-des').first.wait_for(state="visible", timeout=3000)
                has_comments = True
            except:
                print("      [-] Quán chưa có bình luận hoặc không tìm thấy phần đánh giá.")

            if has_comments:
                max_clicks = 3
                clicks = 0
                while clicks < max_clicks:
                    page.evaluate("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)

                    buttons = page.locator('.pn-loadmore a.fd-btn-more, a:has-text("Xem thêm bình luận"), a.btn-load-more').all()
                    
                    target_btn = None
                    for btn in buttons:
                        if btn.is_visible():
                            target_btn = btn
                            break
                    
                    if target_btn:
                        try:
                            target_btn.scroll_into_view_if_needed()
                            time.sleep(0.5)
                            target_btn.click(force=True)
                            clicks += 1
                            print(f"      [+] Đã bấm nút 'Xem thêm' hiển thị trên màn hình thành công lần {clicks}/{max_clicks}")
                            time.sleep(2.5) 
                        except Exception as e:
                            print(f"      [!] Lỗi khi bấm nút: {e}")
                            break
                    else:
                        print("      [-] Không tìm thấy nút 'Xem thêm' hiển thị (đã load hết bình luận).")
                        break

                print(f"      -> Đã bấm {clicks} lần, đang chốt sổ dữ liệu...")
                time.sleep(1.5) 
                
                if len(intercepted_comments) > 0:
                    result['comments'] = intercepted_comments
                    result['comment_scores'] = intercepted_scores
                else:
                    static_comments = []
                    static_scores = []
                    
                    # Tìm thẻ <li> trước để tránh trùng lặp thẻ cha con
                    static_blocks = page.locator('li.review-item').all()
                    if len(static_blocks) == 0:
                        static_blocks = page.locator('div.review-des.fd-clearbox').all()
                    
                    for block in static_blocks:
                        try:
                            block_html = block.inner_html()
                            if '{{Model.' in block_html or 'preload-bar' in block_html:
                                continue
                                
                            desc = "N/A"
                            score = "N/A"
                            
                            # Điểm
                            score_loc = block.locator('[itemprop="ratingValue"], .review-points span').first
                            if score_loc.count() > 0:
                                score_text = score_loc.inner_text().strip()
                                try:
                                    float_score = float(score_text.replace(',', '.'))
                                    normalized = float_score / 2 
                                    # Chỉ lấy 1 số sau dấu phẩy nếu có
                                    normalized = round(normalized, 1)
                                    score = str(int(normalized)) if normalized.is_integer() else str(normalized).replace('.', ',')
                                except:
                                    score = score_text
                                    
                            # Nội dung
                            comment_loc = block.locator('[itemprop="reviewBody"], .rd-des > span').first
                            if comment_loc.count() > 0:
                                desc = comment_loc.inner_text().strip()
                            else:
                                fallback_loc = block.locator('.rd-des').first
                                if fallback_loc.count() > 0:
                                    desc = fallback_loc.inner_text().replace('...Xem thêm', '').replace('Xem thêm', '').strip()
                                    
                            if desc == "": desc = "N/A"
                            
                            # Lọc dữ liệu rác
                            if score == "N/A":
                                continue
                                
                            static_comments.append(desc)
                            static_scores.append(score)
                        except:
                            continue
                            
                    result['comments'] = static_comments
                    result['comment_scores'] = static_scores

                if len(result['comment_scores']) < 20:
                    print(f"DEBUG: {result['comment_scores']}")

        except Exception as e:
            print(f"      [!] Lỗi quá trình xử lý bình luận: {e}")

    except Exception as e:
        print(f"      [!] Lỗi không thể tải hoặc đọc link chi tiết: {e}")
        
    finally:
        try:
            page.remove_listener("response", intercept_api_response)
        except:
            pass
            
    return result


In [ ]:
def get_branches_from_chain_page(page, url, parent_item):
    branches = []
    try:
        page.goto(url, timeout=60000)
        page.wait_for_load_state('domcontentloaded')
        time.sleep(1.5)

        # Cuộn và nhấn nút "Xem thêm" để hiển thị toàn bộ chi nhánh
        last_count = 0
        stuck_count = 0
        while True:
            current_items = page.locator('.ldc-item').count()
            
            if current_items == last_count and current_items > 0:
                stuck_count += 1
                if stuck_count >= 3:
                    break
            else:
                stuck_count = 0
                last_count = current_items

            try:
                load_more_btn = page.locator('.pn-loadmore a.fd-btn-more').first
                if load_more_btn.is_visible(timeout=2000):
                    load_more_btn.click(force=True)
                    time.sleep(1.5)
                else:
                    break
            except:
                break

        # Trích xuất dữ liệu của từng chi nhánh
        items = page.locator('.ldc-item').all()
        for item in items:
            branch = {
                'district': parent_item['district'],
                'score': parent_item.get('score', 'N/A')
            }
            
            try:
                # Lấy URL và Tên chi nhánh
                link_loc = item.locator('.ldc-item-h-name h2 a').first
                branch_url = link_loc.get_attribute('href')
                if branch_url and not branch_url.startswith('http'):
                    branch_url = f"https://www.foody.vn{branch_url}"
                branch['url'] = branch_url
                branch['name'] = link_loc.inner_text().strip()

                # Lấy Địa chỉ chi nhánh
                addr_loc = item.locator('.ldc-item-h-address span').first
                if addr_loc.count() > 0:
                    branch['address'] = addr_loc.inner_text().strip()
                else:
                    branch['address'] = "N/A"

                branches.append(branch)
            except Exception as e:
                print(f"      [!] Lỗi trích xuất dữ liệu chi nhánh cục bộ: {e}")
                continue

    except Exception as e:
        print(f"      [!] Lỗi khi truy cập trang chuỗi thương hiệu: {e}")

    return branches


In [ ]:
def scrape_food():
    all_restaurants = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False, args=["--start-maximized"])
        context = browser.new_context(no_viewport=True)
        page = context.new_page()
        page_detail = context.new_page()

        for district in DISTRICTS:
            print(f"\n{'='*50}")
            print(f"Đang cào tại: {district.upper()}")
            print(f"{'='*50}")
            
            url = f"https://www.foody.vn/ho-chi-minh/dia-diem-tai-{district}"
            
            try:
                page.goto(url, timeout=60000)
                page.wait_for_selector('.filter-result-item', timeout=30000)
            except:
                print(f"   -> Lỗi load trang {district} hoặc không có quán nào. Bỏ qua.")
                continue

            # --- KÉO LOAD MORE ĐỂ LẤY DANH SÁCH ---
            last_count = 0
            stuck_count = 0
            
            while True:
                current_cards = page.locator('.filter-result-item').count()
                print(f"   [Loading] Đang gom được {current_cards} quán tại {district}...")

                if current_cards == last_count and current_cards > 0:
                    stuck_count += 1
                    if stuck_count >= 5: 
                        print("   => Không tải thêm được quán nào sau 5 lần thử, có thể đã hết quán hoặc bị pop-up chặn. Bỏ qua quận này!")
                        break
                else:
                    stuck_count = 0
                    last_count = current_cards

                page.keyboard.press("Escape")
                
                page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                time.sleep(1.5)
                
                try:
                    load_more_btn = page.locator('#scrollLoadingPage a').first
                    if load_more_btn.is_visible():
                        load_more_btn.click(force=True)
                        time.sleep(2)
                        
                        if page.locator('.login-popup').is_visible():
                            print("   => Foody nó bắt đăng nhập. Bỏ qua quận này!")
                            break
                    else:
                        print("   => Hết nút bấm. Cào quận này!")
                        break
                except:
                    break

            # --- VÉT DATA CƠ BẢN TRÊN TRANG TỔNG ---
            print(f"   -> Đang xử lý {current_cards} quán...")
            restaurant_elements = page.locator('.filter-result-item').all()
            
            district_basic_data = []
            for res in restaurant_elements:
                item = {'district': district}
                try:
                    link_locator = res.locator('.resname h2 a')
                    if link_locator.count() == 0:
                        continue 
                        
                    link_element = link_locator.first
                    item['url'] = link_element.get_attribute('href', timeout=1000)
                    if item['url'] and not item['url'].startswith('http'):
                        item['url'] = f"https://www.foody.vn{item['url']}"
                        
                    item['name'] = link_element.inner_text(timeout=1000).strip()
                    
                    # Lấy địa chỉ
                    try:
                        address_container = res.locator('.result-address .address')
                        if address_container.locator('.branch-address').count() > 0:
                            item['address'] = "Nhiều chi nhánh"
                        else:
                            item['address'] = address_container.inner_text(timeout=1000).replace('\n', ', ').strip()
                    except:
                        item['address'] = "N/A"
                        
                    # Lấy điểm số tổng
                    try:
                        item['score'] = res.locator('.point').inner_text(timeout=1000).strip()
                    except:
                        item['score'] = "N/A"

                    print(f"      + Tìm thấy quán: {item['name']} - Địa chỉ: {item['address']}")

                    district_basic_data.append(item)
                except: continue

            # --- LẤY CHI TIẾT CỦA TỪNG QUÁN TRONG QUẬN ---
            print(f"\n   ========== BẮT ĐẦU VÀO TRANG CHI TIẾT CỦA {len(district_basic_data)} MỤC ({district.upper()}) ==========")
            
            valid_district_restaurants = []

            for idx, res_item in enumerate(district_basic_data):
                print(f"  [{idx+1}/{len(district_basic_data)}] Đang xử lý đối tượng: {res_item['name']}")
                
                # Biến lưu trữ các mục cần thu thập chi tiết (có thể là 1 quán lẻ hoặc mảng nhiều chi nhánh)
                items_to_process = []
                
                # Phát hiện nếu đây là trang chuỗi thương hiệu
                if res_item['address'] == "Nhiều chi nhánh" or "/thuong-hieu/" in res_item['url']:
                    print(f"      -> Hệ thống nhận diện đây là chuỗi nhà hàng. Đang tải danh sách các chi nhánh...")
                    branches = get_branches_from_chain_page(page_detail, res_item['url'], res_item)
                    items_to_process.extend(branches)
                    print(f"      -> Tìm thấy {len(branches)} chi nhánh trực thuộc.")
                else:
                    items_to_process.append(res_item)

                # Thu thập thông tin chi tiết cho từng cơ sở cụ thể
                for branch_item in items_to_process:
                    # Nếu trùng url với quán ăn trong danh sách tổng => bỏ qua
                    if any(r['url'] == branch_item['url'] for r in all_restaurants):
                        print(f"      + Quán {branch_item['name']} đã tồn tại trong danh sách tổng, bỏ qua để tránh trùng lặp.")
                        continue

                    if len(items_to_process) > 1:
                        print(f"      -> Thu thập chi tiết chi nhánh: {branch_item['name']}:")
                    else:
                        print(f"      -> Cào chi tiết: {branch_item['name']}:")

                    detail_data = get_details_from_detail_page(page_detail, branch_item['url'])

                    # Bỏ qua quán đã ngưng hoạt động hoặc không còn tương tác
                    if not detail_data['is_active']:
                        continue
                    
                    # Gộp data
                    branch_item['price'] = detail_data['price']
                    branch_item['hours'] = detail_data['hours']
                    
                    for cat_name, cat_score in detail_data['sub_scores'].items():
                        branch_item[f'score_{cat_name}'] = cat_score

                    # Nếu không có điểm tổng, gán giá trị mặc định
                    if 'score' not in detail_data:
                        branch_item['score'] = "0"
                    else:
                        branch_item['score'] = detail_data['score']
                    
                    # Ép list comment thành mảng chuỗi để lưu vào CSV
                    branch_item['comments'] = detail_data['comments'][:MAX_COMMENTS_PER_RESTAURANT]

                    # Lấy điểm của từng bình luận (nếu có) và gán vào branch_item
                    if detail_data.get('comment_scores'):
                        branch_item['comment_scores'] = f"[{'; '.join(detail_data['comment_scores'])}]"
                    else:
                        branch_item['comment_scores'] = "[]"

                    # Lấy số lượng bình luận
                    branch_item['num_comments'] = detail_data['num_comments']
                    
                    if len(items_to_process) > 1:
                        print(f"         + Mức giá: {branch_item['price']}")
                        print(f"         + Điểm tổng: {branch_item['score']} (dựa trên {branch_item['num_comments']} đánh giá)")
                        print(f"         + Số bình luận: {len(branch_item['comments'])}")
                        print(f"         + Debug comment_scores: {len(detail_data['comment_scores']) if detail_data.get('comment_scores') else '0'}")
                        print(f"         + Giờ mở cửa: {branch_item['hours']}")
                    else:
                        print(f"      -> Mức giá: {branch_item['price']}")
                        print(f"      -> Điểm tổng: {branch_item['score']} (dựa trên {branch_item['num_comments']} đánh giá)")
                        print(f"      -> Số bình luận: {len(branch_item['comments'])}")
                        print(f"      -> Debug comment_scores: {len(detail_data['comment_scores']) if detail_data.get('comment_scores') else '0'}")
                        print(f"      -> Giờ mở cửa: {branch_item['hours']}")
                    
                    valid_district_restaurants.append(branch_item)
                    all_restaurants.append(branch_item)

            # LƯU TẠM CSV THEO QUẬN
            save_to_csv(valid_district_restaurants, district)

        browser.close()

    # --- LƯU FILE TỔNG (GỘP 24 QUẬN LẠI) ---
    if len(all_restaurants) > 0:
        df = pd.DataFrame(all_restaurants)
        
        # Lọc sạch trùng lặp do Foody đôi khi hiển thị 1 quán nhiều lần
        df = df.drop_duplicates(subset=['url'])
            
        # Điền giá trị rỗng vào điểm khuyết thiếu
        score_cols = [col for col in df.columns if col.startswith('score_')]
        for col in score_cols:
            df[col] = df[col].fillna('')

        # Tạo cột 'category': tất cả giá trị là 'dining'
        df['category'] = 'dining'

        # Sắp xếp các cột theo thứ tự: district, name, address, score, num_comments, price, category, hours, comments, comment_scores, url, score_*
        ordered_cols = ['district', 'name', 'address', 'score', 'num_comments', 'price', 'category', 'hours', 'comments', 'comment_scores', 'url'] + score_cols
        df = df[ordered_cols]

        df.to_csv(f'tphcm_food_{len(df)}.csv', index=False, encoding='utf-8-sig')
        print(f"\n[HOÀN THÀNH TỔNG] Tổng cộng: {len(df)} quán ăn. Nằm ở file: tphcm_food_{len(df)}.csv")
    else:
        print("Lỗi: Không lấy được data!")


## 4.4.2 Quá trình thực hiện


Quá trình thu thập dữ liệu được thực hiện hoàn toàn tự động thông qua việc mô phỏng hành vi của một người dùng thật đang sử dụng Foody.vn để tìm kiếm và khám phá các nhà hàng/quán ăn tại TP. Hồ Chí Minh. Quy trình gồm 4 giai đoạn chính được lặp lại cho từng quận/huyện:

**Giai đoạn 1: Thu thập danh sách quán từ trang tìm kiếm**
- Bước 1.1: Xây dựng URL tìm kiếm với district slug
- Bước 1.2: Tải trang và chờ items xuất hiện
- Bước 1.3: Vòng lặp scroll + click load more để lấy toàn bộ (Dừng khi `stuck_count >= 5` HOẶC không còn nút load more HOẶC popup login bắt buộc xuất hiện)
- Bước 1.4: Trích xuất thông tin cơ bản từ mỗi item: `name`, `url`, `address`, `score`, `district` (Từ `address` hoặc `url` xác định quán lẻ hay chuỗi cửa hàng)

**Giai đoạn 2: Phân loại quán lẻ vs chuỗi thương hiệu**
- Bước 2.1: Duyệt qua từng item
- Bước 2.2: Phát hiện chuỗi thương hiệu qua 2 dấu hiệu:
  - Điều kiện 1: `address == "Nhiều chi nhánh"`
  - HOẶC Điều kiện 2: URL chứa `/thuong-hieu/`
- Bước 2.3: Quyết định luồng xử lý:
  - Nếu là **quán lẻ**: thêm item vào `items_to_process` (mảng có 1 phần tử)
  - Nếu là **chuỗi**: gọi giai đoạn 3 để lấy chi nhánh

**Giai đoạn 3: Lấy danh sách chi nhánh từ trang thương hiệu** (chỉ áp dụng cho chuỗi)
- Bước 3.1: Gọi `get_branches_from_chain_page(page_detail, url, parent_item)`
  - Input: URL chuỗi thương hiệu, parent_item chứa district và score từ trang tổng
- Bước 3.2: Tải trang thương hiệu
- Bước 3.3: Vòng lặp click "Xem thêm" để hiển thị toàn bộ chi nhánh (Dừng khi `stuck_count >= 3` - không tải thêm được sau 3 lần)
- Bước 3.4: Trích xuất từ mỗi item: `name`, `url`, `address`, `score`, `district` (`district`, `score` được kế thừa từ parent_item)
- Bước 3.5: Thêm tất cả chi nhánh vào `items_to_process`

**Giai đoạn 4: Thu thập thông tin chi tiết từ từng trang quán**
- Bước 4.1: Duyệt qua mỗi item trong `items_to_process`
- Bước 4.2: Gọi `get_details_from_detail_page(page_detail, url)`
- Bước 4.3: Trong hàm get_details:
  - **Bộ lọc API**: Đăng ký listener lắng nghe XHR/Fetch response → tự động bắt dữ liệu review từ API
  - **Kiểm tra quán còn hoạt động**:
    - Tìm `.micro-notification` chứa "chưa có tương tác"
    - Nếu tìm thấy và visible → return `is_active = False` → bỏ qua quán này
  - Lấy điểm tổng (normalize thang 10 → thang 5)
  - Lấy giá
    - **Kiểm tra dữ liệu đủ**: Nếu `score == '0'` VÀ `price == '0'` → quán mới → bỏ qua
  - Lấy điểm hạng mục (`sub_scores`)
  - Lấy giờ mở cửa (`hours`)
  - Lấy số lượng bình luận (`num_comments`)
  - **Lấy bình luận**:
    1. **Cuộn lazy-load**: 20 lần scroll để kích hoạt load comments
    2. Kiểm tra có bình luận (timeout 3s)
    3. Click "Xem thêm" tối đa 3 lần (chờ 2.5s sau mỗi lần)
    4. Ưu tiên dùng dữ liệu API đã intercept; nếu không có → fallback về DOM tĩnh
    5. Bao gồm `comment_scores` (điểm đi kèm từng bình luận)
- Bước 4.4: Kiểm tra `is_active`
  - Nếu False → skip item này, không thêm vào results
- Bước 4.5: Merge dữ liệu (`price`, `sub_scores`, `score`, `hours`, `num_comments`, `comment_scores`, `comments`)
- Bước 4.6: Thêm vào results
  - `valid_district_restaurants.append(branch_item)`
  - `all_restaurants.append(branch_item)`

**Lưu trữ dữ liệu:**
- Lưu file theo quận ngay sau khi hoàn thành mỗi quận: `{district}_Restaurants_{count}.csv` trong `tphcm_foody_data/`
- Lưu file tổng sau khi hoàn thành 24 quận: `tphcm_food.csv`


In [ ]:
if __name__ == '__main__':
    scrape_food()
